# PYTHON TYPE HINTS

*And the Typing System - The Complete Professional Handbook*

From Fundamentals to Data Analytics, Data Science, Machine Learning, and AI Engineering

**Kajola Gbenga**

*Data Scientist & AI Engineer*

---

## Table of Contents

1. Introduction to Type Hints
2. Basic Type Annotations and Built-in Generic Types
3. Optional and Union Types
4. Any, Literal, and Type Aliases
5. TypeVar, Generic Classes, and Generic Functions
6. Callable and Protocol
7. TypedDict, NamedTuple, and Dataclasses
8. ClassVar, Final, and Self
9. Type Narrowing, TypeGuard, Forward References, Recursive Types, and Inference
10. Static Type Checkers, and Type Checking vs Runtime Validation
11. Type Hints Across APIs, Data Analytics, Data Science, ML, and AI Engineering
12. Typing Decorators: ParamSpec, Concatenate, and Annotated
13. Async Typing, Exceptions, NewType, and Overload
14. Type Hints, OOP, and Design Patterns
15. Professional Project Structure, Common Mistakes, cast(), and Best Practices
16. Real-World Projects (1 of 2): Student System, Sales Pipeline, Financial Processor, ETL
17. Real-World Projects (2 of 2): ML Preprocessing, REST Client, NLP Pipeline, AI/LLM App
18. 60 Interview Questions With Answers
19. 50 Practical Exercises With Full Solutions
20. Final Type-Hint Cheat Sheet

---

### How to Use This Notebook

Every code example is a **live, runnable code cell**. Run each with `Shift+Enter`.

Static type checking itself (Mypy/Pyright) cannot run inside a plain code cell's Python kernel - it is a separate command-line tool. Where this notebook says 'a type checker would flag this', that is a STATIC-time claim, not something the Python kernel itself will show you when you run the cell. A few cells demonstrate this gap directly: they run without error in the kernel despite being annotated incorrectly - that contrast is the entire point of Part 1 and Part 10.

You can install and run a real type checker against any cell's code by saving it to a `.py` file, for example:

In [ ]:
%%writefile example.py
def greet(name: str) -> str:
    return "Hello, " + name

greet(42)

In [ ]:
!pip install mypy --quiet
!mypy example.py

## Part 1 - Introduction to Type Hints

### 1.1 What Type Hints Are

A type hint is an optional piece of syntax attached to a variable, function parameter, or return value that states the intended type of that value. Type hints were introduced in PEP 484 and have expanded steadily ever since (PEP 526, 585, 604, 612, 646, 673, 695, among others). They look like ordinary Python syntax but carry no runtime behavior of their own - they exist purely as metadata that tools (type checkers, IDEs, documentation generators) can read and reason about.

In [ ]:
def calculate_total(price: float, quantity: int) -> float:
    return price * quantity

Here, price: float and quantity: int are parameter annotations, and -> float is the return annotation. None of this changes how the function executes - Python will happily call calculate_total("5", "3") and concatenate strings instead of multiplying numbers, with no error raised by the annotations themselves.

### 1.2 Why Python Is Dynamically Typed

Python decides the type of a variable at runtime, based on the object currently bound to that name, rather than requiring a fixed type declaration ahead of time (as in Java or C). The same variable name can be rebound to values of entirely different types over its lifetime. This flexibility is a major reason Python is fast to write and popular for scripting, data work, and prototyping - but it also means that type-related mistakes that a compiler would catch in a statically-typed language surface only when the offending line actually executes, sometimes deep in production.

### 1.3 Static vs Dynamic Typing

| Aspect | Static Typing | Dynamic Typing (Python's default) |
| --- | --- | --- |
| When types are checked | Before running the program (compile time) | While the program runs |
| Where errors surface | Immediately, at the point of the mismatch | Only when the mismatched line executes |
| Typical languages | Java, C, Rust, TypeScript (checked) | Python, JavaScript, Ruby (by default) |
| Python's relationship | Achievable via optional type hints + a checker | The language's native runtime behavior |

Type hints let Python have the best of both worlds: the language itself stays dynamically typed at runtime, while an external static type checker (Part 10) can analyze your annotated code before it ever runs, catching whole categories of bugs early - without changing Python's execution model at all.

### 1.4 Strong vs Weak Typing

This is a separate axis from static/dynamic. A strongly typed language refuses to silently coerce incompatible types (Python will not add 5 + "5" - it raises a TypeError). A weakly typed language performs such coercions automatically (JavaScript's 5 + "5" produces "55"). Python is dynamically typed but strongly typed: it decides types at runtime, but it is strict about not mixing incompatible types silently.

### 1.5 Type Hints vs Type Enforcement

This is the single most important idea in this entire handbook, and it is worth stating as plainly as possible: type hints are not enforced by the Python interpreter. They are read by nobody at runtime except tools you deliberately invoke - your IDE, a linter, or a static type checker such as Mypy or Pyright.

In [ ]:
def greet(name: str) -> str:
    return "Hello, " + name

print(greet(42))   # Runs WITHOUT error at the interpreter level...
                   # ...then crashes with TypeError inside the function,
                   # because 'str' + int is not allowed - but that TypeError
                   # comes from the + operator, NOT from the annotation.

> **Critical Distinction:** Python's interpreter does not consult annotations to validate arguments, ever, in any version. A static type checker run separately (e.g. `mypy script.py`) is what would have told you, before running anything, that greet(42) passes an int where str was expected.

### 1.6 Why Python Does Not Enforce Annotations at Runtime

- Backwards compatibility: annotations were added in PEP 3107 (2006, Python 3.0) purely as syntax, with semantics left open - enforcing them retroactively would have broken enormous amounts of existing code.

- Performance: checking every function call against declared types would add runtime overhead to every single call in every Python program, even ones that never wanted type checking.

- Flexibility: Python's philosophy favors optional, additive tooling over mandatory language changes - you opt in to static checking by running a checker, rather than the language forcing it on you.

- Gradual typing: annotations are designed to be added incrementally to an existing, unannotated codebase (see Part 40), which is only practical if adding a hint cannot suddenly start raising new runtime errors.

### 1.7 Benefits of Type Hints

- Catch bugs before running the code, when checked by a static type checker in your editor or CI pipeline.

- Serve as always-up-to-date, enforced-by-tooling documentation of what a function expects and returns.

- Enable much richer autocomplete and inline documentation in modern IDEs (VS Code, PyCharm).

- Make large codebases and teams safer to refactor, since a type checker will flag every call site broken by a signature change.

- Improve the design of function and class interfaces, because writing an honest annotation often exposes an overly vague or overloaded API.

### 1.8 Limitations of Type Hints

- They provide zero runtime protection on their own - Part 24 covers exactly where real validation (e.g. Pydantic) is still needed.

- Some dynamic Python patterns (heavy use of **kwargs, monkey-patching, metaclass magic) are difficult or impossible to express precisely.

- Third-party libraries may have incomplete, outdated, or overly loose type stubs, producing false confidence or noisy false positives (Part 26).

- Overly complex generic types can make code harder to read than the untyped version, defeating the readability benefit (Part 41).

### 1.9 Type Hints in Professional Python Development

In modern professional Python - data platforms, ML services, backend APIs, AI agent frameworks - type hints are now the default expectation, not an optional flourish. Popular frameworks (FastAPI, Pydantic, SQLAlchemy 2.0, Django's newer APIs) are built directly around annotations, using them to drive validation, serialization, and even database schema generation. Static type checking is commonly a required, blocking step in CI/CD pipelines (Part 40), on par with unit tests and linting.

## Part 2 - Basic Type Annotations and Built-in Generic Types

### 2.1 Variable Annotations

A variable annotation states the intended type of a name, optionally alongside an initial value. This syntax comes from PEP 526.

In [ ]:
name: str = "Ada"
age: int = 34
price: float = 19.99
is_active: bool = True

# Annotation without a value yet (declares intent, no binding happens):
score: int

> **Common Misunderstanding:** score: int on its own does NOT create the variable - it only records, for the type checker, that whenever 'score' is later assigned in this scope, it should hold an int. Referencing score before an actual assignment still raises a NameError at runtime.

### 2.2 Function Parameter and Return Annotations

In [ ]:
def calculate_total(price: float, quantity: int) -> float:
    return price * quantity

def full_name(first: str, last: str) -> str:
    return f"{first} {last}"

Line by line: price: float and quantity: int tell readers and checkers what types the caller should pass. -> float, placed after the closing parenthesis and before the colon, declares the type the function returns. If a function has no annotation at all, both humans and type checkers treat every parameter and the return value as implicitly Any (Part 6) - unannotated, unchecked.

### 2.3 None Return Types

A function that performs an action but does not return a meaningful value (its implicit return is None) should be annotated -> None. This is not the same as omitting the annotation - omitting it means unchecked, whereas -> None explicitly documents 'this function's return value should never be used.'

In [ ]:
def log_event(message: str) -> None:
    print(f"[LOG] {message}")
    # no return statement - implicitly returns None

### 2.4 Annotating Constants

Module-level constants are commonly annotated for clarity, and combined with Final (Part 16) when they must never be reassigned.

In [ ]:
MAX_RETRIES: int = 5
API_BASE_URL: str = "https://api.example.com"

### 2.5 Annotating Class Attributes

In [ ]:
class User:
    name: str          # class-level annotation, no value - just documents the type
    signup_bonus: int = 100   # class attribute WITH a default value, shared by all instances

    def __init__(self, name: str, age: int) -> None:
        self.name = name   # instance attribute, type inferred from the annotated parameter
        self.age: int = age   # explicit instance-attribute annotation (also acceptable)

### 2.6 Annotating Local Variables

Local variables inside a function usually do not need an explicit annotation, because a type checker can infer the type from the assigned value (see Part 22, Type Inference). An explicit annotation is most useful when the initial value does not make the intended type obvious - typically an empty collection.

In [ ]:
def collect_even_numbers(limit: int) -> list[int]:
    results: list[int] = []   # without the annotation, an empty list gives no type clue
    for n in range(limit):
        if n % 2 == 0:
            results.append(n)
    return results

## Part 2 (continued) - Built-in Generic Types

Since Python 3.9 (PEP 585), the built-in collection types themselves - list, dict, tuple, set, frozenset - can be parameterized directly with square brackets, without importing anything from the typing module. This is now the preferred, modern syntax.

### 2.7 list[T]

In [ ]:
names: list[str] = ["Ada", "Bo", "Cy"]
prices: list[float] = [9.99, 14.50, 3.25]

### 2.8 dict[K, V]

In [ ]:
ages: dict[str, int] = {"Ada": 34, "Bo": 22}
inventory: dict[str, float] = {"widget": 4.99, "gadget": 12.50}

### 2.9 tuple[...]

Unlike list and dict, tuple has two distinct annotation styles depending on intent: a fixed-length tuple lists each position's type explicitly; a variable-length homogeneous tuple uses ... (an actual Ellipsis literal, not three dots meaning 'etc').

In [ ]:
point: tuple[int, str] = (1, "north")            # exactly 2 elements: int, then str
coordinates: tuple[float, float, float] = (1.0, 2.0, 3.0)  # exactly 3 floats
scores: tuple[int, ...] = (95, 88, 76, 100)       # ANY number of ints - the '...' means 'more of the same'

### 2.10 set[T] and frozenset[T]

In [ ]:
unique_tags: set[str] = {"python", "typing", "mypy"}
allowed_roles: frozenset[str] = frozenset({"admin", "editor", "viewer"})

### 2.11 Modern Syntax vs Legacy typing.List / typing.Dict

Before Python 3.9, the built-in list/dict/tuple/set could not be parameterized directly, so the typing module provided capitalized equivalents (List, Dict, Tuple, Set, FrozenSet) purely so they could carry type parameters. That legacy syntax still works today (for backward compatibility) but is considered outdated - modern code should use the lowercase built-ins.

| Legacy (pre-3.9, still works) | Modern (3.9+, preferred) |
| --- | --- |
| from typing import List, Dict, Tuple, Set | No import needed |
| names: List[str] | names: list[str] |
| ages: Dict[str, int] | ages: dict[str, int] |
| point: Tuple[int, str] | point: tuple[int, str] |
| tags: Set[str] | tags: set[str] |

> **Version Note:** If your project must support Python 3.8 or earlier, the legacy typing.List-style imports are still required. For any project targeting 3.9+, use the lowercase built-ins - this is the standard professional convention today.

### 2.12 Nested Collections

Generic types compose freely, which is what makes them expressive enough for realistic data shapes.

#### Dictionaries Containing Lists

In [ ]:
students_by_grade: dict[str, list[str]] = {
    "A": ["Ada", "Cy"],
    "B": ["Bo"],
}

#### Lists Containing Dictionaries

In [ ]:
users: list[dict[str, str]] = [
    {"name": "Ada", "role": "admin"},
    {"name": "Bo", "role": "viewer"},
]

#### Complex Nested Structures

In [ ]:
# A realistic API-response shape: a dict of report names, each holding
# a list of (date, value) tuples.
reports: dict[str, list[tuple[str, float]]] = {
    "sales": [("2026-01-01", 1200.50), ("2026-01-02", 980.25)],
    "refunds": [("2026-01-01", 45.00)],
}

### Part 2 Common Mistakes and Best Practices

- Mistake: writing tuple[int] when a variable-length tuple of ints was intended - tuple[int] means exactly one int; the correct annotation is tuple[int, ...].

- Mistake: mixing legacy typing.List and modern list[...] inconsistently within the same codebase - pick one convention (modern, for 3.9+ projects) and apply it uniformly.

- Best practice: annotate empty-collection local variables explicitly, since inference has nothing to work from.

- Best practice: keep nesting to two or three levels deep at most - beyond that, define a TypeAlias (Part 4) or a TypedDict/dataclass (Parts 13, 15) instead of an unreadable wall of brackets.

## Part 3 - Optional and Union Types

### 3.1 What Optional Means

Optional[X] means 'this value is either of type X, or it is None.' It is exactly equivalent to Union[X, None] (see 3.5) - Optional is purely a shorthand for the extremely common 'value or nothing' pattern, not a distinct concept of its own.

In [ ]:
from typing import Optional

def find_user(user_id: int) -> Optional[str]:
    users = {1: "Ada", 2: "Bo"}
    return users.get(user_id)   # returns None if user_id is not a key

### 3.2 Modern Syntax: X | None

Since Python 3.10 (PEP 604), the pipe operator | can build union types directly, without importing anything. X | None is the modern, preferred equivalent of Optional[X].

In [ ]:
def find_user(user_id: int) -> str | None:
    users = {1: "Ada", 2: "Bo"}
    return users.get(user_id)

> **Modern vs Legacy:** Optional[X] and X | None mean exactly the same thing to a type checker. Optional[X] still requires `from typing import Optional`; X | None needs no import at all on Python 3.10+. New code targeting 3.10+ should prefer X | None.

### 3.3 Why None Needs Special Consideration

None is an extremely common return value and default, and forgetting to account for it is one of the single most common sources of AttributeError and TypeError bugs in real Python code ('NoneType has no attribute ...'). Explicitly annotating a value as X | None forces both the author and a static checker to confront every place that value is used, and flags any usage that does not first handle the None case.

### 3.4 Default Values and Optional Parameters

In [ ]:
def greet(name: str, title: str | None = None) -> str:
    if title is None:
        return f"Hello, {name}"
    return f"Hello, {title} {name}"

> **Common Mistake:** A parameter with default value None must be annotated str | None (or Optional[str]), never just str - annotating it as plain str is a common and genuinely incorrect mistake, since the actual default violates the plain str annotation.

### 3.5 Handling Potentially Missing Values and Type Narrowing

Type narrowing is what a type checker does when it follows a conditional check and concludes that, inside one specific branch, a value's type is more specific than its declared annotation. Checking is None (or is not None) is the standard way to narrow an Optional value down to its non-None type.

In [ ]:
def describe_user(name: str | None) -> str:
    if name is None:
        return "Unknown user"
    # Inside this branch, the type checker now treats 'name' as plain 'str',
    # not 'str | None' - this is narrowing in action.
    return name.upper()

Full coverage of narrowing techniques (isinstance, issubclass, assert, TypeGuard) appears in Part 9.

## Part 3 (continued) - Union Types

### 3.6 Union and the Modern | Operator

Union[A, B, C] (or, equivalently, A | B | C) means 'this value could be any one of these types.' Optional[X] is the special two-member case where the second member is always None.

In [ ]:
from typing import Union

def process(value: Union[int, float]) -> float:
    return float(value)

# Modern equivalent, Python 3.10+:
def process(value: int | float) -> float:
    return float(value)

### 3.7 Multiple Possible Types and Nested Unions

In [ ]:
def parse_id(raw: str | int | None) -> int | None:
    if raw is None:
        return None
    return int(raw)

# Nested unions automatically 'flatten' - this is identical to str | int | None:
Combined = (str | int) | None

### 3.8 When Unions Are Appropriate

- A value genuinely, legitimately can be one of a small, closed set of distinct types (e.g. a JSON-decoded value that could be a dict or a list).

- A function accepts either a raw value or None to represent 'not provided' (Optional).

- An identifier could arrive as either a string or an integer from different callers, and the function normalizes it internally.

### 3.9 When a Union Indicates Poor API Design

A union with many unrelated members, or one that forces every caller to write an isinstance check before doing anything useful, is often a sign that the function is doing too many different jobs, or that the caller should be passing a properly modeled object (a TypedDict, dataclass, or Protocol) instead of loosely unioned primitives.

In [ ]:
# A warning sign: a sprawling, poorly designed union
def handle(payload: dict | list | str | int | float | bool | None) -> None:
    ...   # this signature communicates almost nothing useful to the caller

# Better: model the actual expected shape explicitly (see TypedDict, Part 13)
class EventPayload(TypedDict):
    event_type: str
    data: dict[str, str]

### Part 3 Common Mistakes and Best Practices

- Mistake: giving a parameter a default of None while annotating it as a plain, non-Optional type.

- Mistake: accessing attributes/methods on an Optional value without narrowing it first (forgetting the is None check).

- Best practice: prefer X | None over Optional[X] in any codebase targeting Python 3.10+, for consistency and to avoid an extra import.

- Best practice: if a function returns a wide, unrelated union, consider whether the function's responsibility should be split into two more specific functions instead.

## Part 4 - Any, Literal, and Type Aliases

### 4.1 What Any Means

typing.Any is the deliberate 'escape hatch' of the type system. A value annotated Any is compatible with every other type in both directions: you can assign an Any-typed value to a variable of any type, and you can assign a value of any type to an Any-typed variable, with no complaint from the type checker whatsoever.

In [ ]:
from typing import Any

def process(data: Any) -> Any:
    return data.whatever_attribute_you_like   # no error reported, even if wrong

### 4.2 Why Any Disables Much Static Type Checking

Once a value is Any, the type checker stops verifying anything about how it is used - method calls, attribute access, arithmetic, indexing, all pass silently, because Any is compatible with everything by definition. Any is 'contagious': assigning an Any value into a new variable, or returning it from a function, spreads the untyped-ness onward, unless a later line narrows or re-annotates it.

In [ ]:
def load_config() -> Any:
    import json
    return json.load(open("config.json"))

config = load_config()
print(config.nonexistent_field.also_fake())   # type checker: no complaints at all

### 4.3 Appropriate Uses of Any

- Interfacing with genuinely untyped or dynamically shaped external data (raw JSON before it is validated/parsed into a proper model).

- Gradually migrating a large legacy codebase, where Any is a deliberate, temporary placeholder (see 4.5) rather than a permanent choice.

- Extremely generic utility functions where no useful constraint on the input actually exists (rare in practice - object is often a better, still-type-checked choice; see the note below).

> **Any vs object:** object is frequently a better choice than Any when you want to say 'this could be anything' but still want the type checker to require an isinstance check before you use the value in a specific way. Unlike Any, object is NOT compatible with arbitrary attribute access - the checker will flag misuse.

### 4.4 Dangerous Uses of Any

- Annotating a function's return type as Any 'just to make the checker stop complaining' - this silently defeats the purpose of typing everywhere that return value later flows.

- Using Any as a substitute for genuinely modeling a complex-but-knowable structure (a JSON API response) that could instead be a TypedDict or dataclass (Parts 13, 15).

- Sprinkling Any throughout a codebase to silence errors during a deadline crunch, with no plan to remove it - Any debt accumulates exactly like technical debt.

### 4.5 Migrating Poorly Typed Code Toward Better Annotations

A common, healthy pattern for legacy code is to start every previously-unannotated function with Any (or leave it unannotated, which a checker treats the same way in non-strict mode), then incrementally tighten each signature toward its real type as you touch that code, verified by re-running the checker after each change. Part 40 covers gradual typing at the project level.

In [ ]:
# Step 1 (starting point, honestly documents current uncertainty):
def transform(data: Any) -> Any:
    ...

# Step 2 (after investigating the actual shape of 'data'):
def transform(data: list[dict[str, str]]) -> dict[str, int]:
    ...

## Part 4 (continued) - Literal

### 4.6 What Literal Restricts

typing.Literal narrows an annotation down to one or more specific, exact values, rather than an entire type. Literal["GET", "POST"] means 'this must be exactly the string "GET" or exactly the string "POST" - nothing else, even though both are technically str.'

In [ ]:
from typing import Literal

def set_log_level(level: Literal["DEBUG", "INFO", "WARNING", "ERROR"]) -> None:
    print(f"Log level set to {level}")

set_log_level("INFO")     # OK
set_log_level("VERBOSE")  # Type checker error: not one of the allowed literals

### 4.7 Literal Integers and Other Literal Values

In [ ]:
from typing import Literal

def set_dice_value(value: Literal[1, 2, 3, 4, 5, 6]) -> None:
    ...

Mode = Literal["read", "write", "append"]

def open_file(path: str, mode: Mode) -> None:
    ...

### 4.8 Practical API and Configuration Examples

In [ ]:
from typing import Literal, TypedDict

class ServerConfig(TypedDict):
    environment: Literal["development", "staging", "production"]
    log_level: Literal["DEBUG", "INFO", "WARNING", "ERROR"]

def deploy(config: ServerConfig) -> None:
    if config["environment"] == "production":
        print("Deploying with extra safety checks...")

Literal is especially valuable for HTTP methods, configuration modes, database isolation levels, and any 'small closed set of valid strings/ints' scenario - it catches typos like "prod" instead of "production" at type-check time, something a plain str annotation could never catch.

## Part 4 (continued) - TypeAlias and Type Aliases

### 4.9 Creating Reusable Aliases

A type alias gives a complex or frequently repeated type expression a short, meaningful name, improving readability without changing behavior at all.

In [ ]:
from typing import TypeAlias

# Legacy/explicit style (works on all supported versions):
UserId: TypeAlias = int
JsonDict: TypeAlias = dict[str, "JsonValue"]

### 4.10 Modern type Statement (Python 3.12+)

Python 3.12 introduced a dedicated type statement (PEP 695), using type as a soft keyword, which is now the preferred way to declare an alias - it is clearer, and it supports generic aliases directly.

In [ ]:
type UserId = int
type JsonDict = dict[str, "JsonValue"]

# A generic alias (Python 3.12+):
type Pair[T] = tuple[T, T]

def swap[T](pair: Pair[T]) -> Pair[T]:
    a, b = pair
    return (b, a)

### 4.11 Aliasing Complex Types

In [ ]:
type ApiResponse = dict[str, str | int | list[str]]
type Handler = "Callable[[str], None]"   # forward reference, see Part 20

### 4.12 When Aliases Improve Readability

- A type expression is used in more than two or three places across a module.

- A nested generic type (dict[str, list[tuple[int, str]]]) would otherwise be repeated verbatim and is hard to read inline.

- A domain-specific name (UserId, Timestamp, JsonValue) communicates intent far better than the raw underlying type does.

### Part 4 Common Mistakes and Best Practices

- Mistake: reaching for Any as a default whenever a type is momentarily unclear, instead of taking a minute to look up the real shape.

- Mistake: using a plain str annotation where the actual valid values form a small closed set - Literal would have caught invalid values.

- Best practice: use Literal for configuration flags, modes, and enum-like string/int values instead of a bare str or int.

- Best practice: reach for a type alias (or the modern type statement on 3.12+) once a compound type expression is repeated three or more times.

## Part 5 - TypeVar, Generic Classes, and Generic Functions

### 5.1 What a Type Variable Is

A type variable is a placeholder that lets a function or class be written once, generically, while still preserving an exact relationship between the types of its inputs and outputs for any specific call. Without one, you either lose that relationship (falling back to Any) or must write a separate function for every concrete type.

In [ ]:
# Without a type variable - the relationship between input and output is LOST:
def first_item(items: list) -> object:
    return items[0]

result = first_item([1, 2, 3])   # type checker only knows 'object', not 'int'

### 5.2 Creating a TypeVar (Legacy Style)

In [ ]:
from typing import TypeVar

T = TypeVar("T")

def first_item(items: list[T]) -> T:
    return items[0]

result = first_item([1, 2, 3])       # type checker infers T = int, result: int
name = first_item(["a", "b", "c"])  # T = str, name: str

Line by line: T = TypeVar("T") declares a reusable type placeholder named T. list[T] -> T tells the checker: 'whatever concrete type fills in T for the list argument, that same concrete type is exactly what comes back out.' This relationship is what makes generics powerful - it is preserved per call, not fixed once for the whole function.

### 5.3 Modern Generic Syntax (Python 3.12+)

PEP 695 (Python 3.12) allows type parameters to be declared directly in square brackets on the function or class, without a separate TypeVar object at all.

In [ ]:
def first_item[T](items: list[T]) -> T:
    return items[0]

> **Modern vs Legacy:** The modern [T] syntax and the legacy TypeVar("T") approach are equivalent to a type checker. Projects targeting Python 3.12+ exclusively should prefer the modern syntax for its brevity; projects supporting older versions must still use TypeVar.

### 5.4 Constraints

A constrained TypeVar restricts T to one of an explicit, closed list of types (not their subclasses in the general sense that bounds allow - constraints mean 'exactly one of these listed types').

In [ ]:
from typing import TypeVar

StrOrInt = TypeVar("StrOrInt", str, int)

def double(value: StrOrInt) -> StrOrInt:
    return value * 2   # works for str (repeats) and int (multiplies) alike

double("ab")   # 'abab' - OK, T = str
double(5)      # 10    - OK, T = int
double(3.5)    # Type checker error - float is not in the constraint list

### 5.5 Bounds

A bounded TypeVar restricts T to a specific type or any of its subclasses - useful when you want to accept 'anything that IS-A X', including future subclasses you have not written yet.

In [ ]:
from typing import TypeVar

class Animal:
    def make_sound(self) -> str:
        return "..."

class Dog(Animal):
    def make_sound(self) -> str:
        return "Woof"

TAnimal = TypeVar("TAnimal", bound=Animal)

def announce(animal: TAnimal) -> TAnimal:
    print(animal.make_sound())
    return animal

announce(Dog())   # T = Dog (a subclass of Animal) - return type is still Dog, not just Animal

| Feature | Constraints | Bounds |
| --- | --- | --- |
| Syntax | TypeVar("T", TypeA, TypeB) | TypeVar("T", bound=TypeA) |
| Meaning | T must be exactly one of the listed types | T must be TypeA or any subclass of it |
| Typical use | A small, fixed, unrelated set of types | An open hierarchy of related classes |

### 5.6 Preserving Relationships Between Input and Output Types

This is the entire point of generics: a function like def identity(x: T) -> T guarantees, for every individual call, that whatever specific type goes in is exactly what comes out - a guarantee Any could never make, since Any provides no such link between parameters.

## Part 5 (continued) - Generic Classes and Functions

### 5.7 Generic Containers and Classes

In [ ]:
from typing import TypeVar, Generic

T = TypeVar("T")

class Stack(Generic[T]):
    def __init__(self) -> None:
        self._items: list[T] = []

    def push(self, item: T) -> None:
        self._items.append(item)

    def pop(self) -> T:
        return self._items.pop()

int_stack: Stack[int] = Stack()
int_stack.push(5)
int_stack.push("oops")   # Type checker error: expected int, got str

Line by line: class Stack(Generic[T]) declares that Stack is generic over one type parameter, T. Every method that mentions T (push, pop) is tied to whichever concrete type was used when the instance was created - Stack[int] locks T to int for that specific instance, and the checker enforces it on every subsequent call.

### 5.8 Modern Generic Class Syntax (Python 3.12+)

In [ ]:
class Stack[T]:
    def __init__(self) -> None:
        self._items: list[T] = []

    def push(self, item: T) -> None:
        self._items.append(item)

    def pop(self) -> T:
        return self._items.pop()

### 5.9 Multiple Type Parameters

In [ ]:
from typing import TypeVar, Generic

K = TypeVar("K")
V = TypeVar("V")

class Cache(Generic[K, V]):
    def __init__(self) -> None:
        self._data: dict[K, V] = {}

    def set(self, key: K, value: V) -> None:
        self._data[key] = value

    def get(self, key: K) -> V | None:
        return self._data.get(key)

user_cache: Cache[int, str] = Cache()   # K=int, V=str

### 5.10 Real-World Example: A Typed Repository

In [ ]:
from typing import TypeVar, Generic

T = TypeVar("T")

class Repository(Generic[T]):
    """A minimal in-memory repository generic over the stored entity type."""
    def __init__(self) -> None:
        self._items: dict[int, T] = {}
        self._next_id = 1

    def add(self, item: T) -> int:
        item_id = self._next_id
        self._items[item_id] = item
        self._next_id += 1
        return item_id

    def get(self, item_id: int) -> T | None:
        return self._items.get(item_id)

class User:
    def __init__(self, name: str) -> None:
        self.name = name

user_repo: Repository[User] = Repository()
new_id = user_repo.add(User("Ada"))
found = user_repo.get(new_id)   # type checker knows: found is 'User | None'

### Part 5 Common Mistakes and Best Practices

- Mistake: using Generic[T] but never actually using T anywhere meaningful in the class body - if a class doesn't need to preserve a type relationship, it likely does not need to be generic.

- Mistake: confusing constraints with bounds, and reaching for a constrained TypeVar when an open, inheritance-friendly bound was actually intended.

- Best practice: name single, unconstrained type variables T, and give constrained/bounded ones a more descriptive name (TAnimal, StrOrInt) so their restriction is self-documenting.

- Best practice: prefer the modern [T] syntax for any project that has already committed to Python 3.12+.

## Part 6 - Callable and Protocol

### 6.1 Annotating Functions Passed as Arguments: Callable

Callable[[ParamTypes], ReturnType] describes a function (or anything else callable, like an object with __call__) that takes parameters of the listed types, in that order, and returns the given type.

In [ ]:
from typing import Callable

def apply_twice(func: Callable[[int], int], value: int) -> int:
    return func(func(value))

def increment(x: int) -> int:
    return x + 1

print(apply_twice(increment, 5))   # 7

Line by line: Callable[[int], int] means 'a callable that takes exactly one int parameter and returns an int.' The outer brackets belong to Callable itself; the inner [int] is the parameter-type list, and the second int (outside those inner brackets) is the return type.

### 6.2 Multiple Parameters and No Parameters

In [ ]:
from typing import Callable

Adder = Callable[[int, int], int]           # takes two ints, returns an int
NoArgCallback = Callable[[], None]          # takes nothing, returns nothing

def combine(a: int, b: int, operation: Adder) -> int:
    return operation(a, b)

### 6.3 Callable[..., ReturnType] for Arbitrary Parameters

When the exact parameter list does not matter (or cannot reasonably be pinned down), Callable[..., ReturnType] uses a literal Ellipsis to mean 'any parameters at all, but this specific return type.' This trades away parameter-checking precision - Part 12's ParamSpec is the more precise tool when parameters genuinely need to be preserved.

In [ ]:
from typing import Callable

def run_and_log(func: Callable[..., int]) -> int:
    result = func()
    print(f"Result was {result}")
    return result

### 6.4 Higher-Order Functions and Callbacks

In [ ]:
from typing import Callable

def retry(
    operation: Callable[[], str],
    on_failure: Callable[[Exception], None],
    attempts: int = 3,
) -> str | None:
    for _ in range(attempts):
        try:
            return operation()
        except Exception as error:
            on_failure(error)
    return None

### 6.5 Functional Programming Applications

In [ ]:
from typing import Callable

def compose(f: Callable[[int], int], g: Callable[[int], int]) -> Callable[[int], int]:
    def composed(x: int) -> int:
        return f(g(x))
    return composed

add_one = lambda x: x + 1
double = lambda x: x * 2
add_then_double = compose(double, add_one)
print(add_then_double(3))   # (3 + 1) * 2 = 8

### 6.6 Decorators and Type Hints (Preview)

A basic decorator can be typed with Callable, but preserving the exact original signature through a decorator requires ParamSpec, covered in full in Part 12 alongside decorators specifically.

In [ ]:
from typing import Callable

def logged(func: Callable[..., int]) -> Callable[..., int]:
    def wrapper(*args: object, **kwargs: object) -> int:
        print(f"Calling {func.__name__}")
        return func(*args, **kwargs)
    return wrapper

## Part 6 (continued) - Protocol

### 6.7 What a Protocol Is

A Protocol defines a structural type: a set of methods and/or attributes that a class must provide, without that class needing to inherit from the Protocol at all. This is Python's static-typing counterpart to duck typing - 'if it walks like a duck and quacks like a duck, it's a duck' - now checkable by a static analyzer rather than only discoverable at runtime.

### 6.8 Structural vs Nominal Typing

| Aspect | Nominal Typing (normal class/inheritance) | Structural Typing (Protocol) |
| --- | --- | --- |
| Compatibility determined by | Explicit inheritance ("is-a" declared in code) | Having the right shape (methods/attributes present) |
| Must subclass? | Yes | No - any class with matching methods qualifies |
| Typical Python tool | class Dog(Animal): | class Speaker(Protocol): def speak(self) -> str: ... |

### 6.9 Defining a Protocol

In [ ]:
from typing import Protocol

class Speaker(Protocol):
    def speak(self) -> str:
        ...

class Dog:               # NOTE: does not inherit from Speaker at all
    def speak(self) -> str:
        return "Woof"

class Robot:              # also does not inherit from Speaker
    def speak(self) -> str:
        return "BEEP"

def announce(speaker: Speaker) -> None:
    print(speaker.speak())

announce(Dog())     # OK - Dog has a matching speak() method
announce(Robot())   # OK - Robot also has a matching speak() method

Line by line: class Speaker(Protocol) declares the required shape - a method speak() returning str, with ... as its placeholder body since Protocol methods are never actually called. Neither Dog nor Robot mentions Speaker anywhere, yet both satisfy it structurally, and a static type checker accepts both calls to announce().

### 6.10 Duck Typing With Static Checking

Protocol gives you the flexibility Python developers have always relied on (accepting 'anything with a .read() method', for example) while still letting a static type checker catch a genuinely incompatible object before runtime - something pure duck typing alone cannot do.

### 6.11 When Protocol Is Better Than Inheritance

- You need to accept objects from third-party libraries that cannot be modified to inherit from your base class.

- You want to decouple a function's requirements from a specific class hierarchy, so callers can supply any compatible object, including simple test doubles.

- You are defining an interface for something with no shared implementation to offer (Protocol has no method bodies to inherit; it purely documents a required shape).

In [ ]:
from typing import Protocol

class SupportsClose(Protocol):
    def close(self) -> None:
        ...

def cleanup(resource: SupportsClose) -> None:
    resource.close()

# Works for file objects, database connections, custom classes -
# anything with a close() method, with zero inheritance required.

### Part 6 Common Mistakes and Best Practices

- Mistake: reaching for Callable[..., Any] out of habit when the actual parameter and return types are perfectly knowable.

- Mistake: building a deep inheritance hierarchy purely to satisfy a type signature, when a lightweight Protocol would have decoupled the code more cleanly.

- Best practice: define small, focused Protocols (one or two methods) rather than one giant Protocol mirroring an entire class - this mirrors the interface-segregation principle from object-oriented design.

- Best practice: use Callable for simple function-shaped parameters, and Protocol once you need more than a single call signature (multiple methods, or attributes plus methods).

## Part 7 - TypedDict, NamedTuple, and Dataclasses

### 7.1 Typed Dictionaries

TypedDict describes a dictionary with a fixed, known set of string keys, each with its own specific value type - unlike dict[str, X], where every value must share the same type X. It exists purely for the type checker; at runtime, a TypedDict instance is a completely ordinary dict.

In [ ]:
from typing import TypedDict

class Movie(TypedDict):
    title: str
    year: int
    rating: float

inception: Movie = {"title": "Inception", "year": 2010, "rating": 8.8}
print(type(inception))   # <class 'dict'> - it really is just a dict at runtime

### 7.2 Required and Optional Fields

By default, every field in a TypedDict is required. Individual fields can be marked optional with NotRequired (Python 3.11+), or an entire TypedDict can be made total=False to make every field optional at once.

In [ ]:
from typing import TypedDict, NotRequired

class Movie(TypedDict):
    title: str
    year: int
    rating: NotRequired[float]   # this key may be omitted entirely

m1: Movie = {"title": "Dune", "year": 2021}                 # OK - rating omitted
m2: Movie = {"title": "Dune", "year": 2021, "rating": 8.0}   # also OK

In [ ]:
# Pre-3.11 alternative: split into a required base and an optional extension
class MovieBase(TypedDict):
    title: str
    year: int

class Movie(MovieBase, total=False):
    rating: float   # optional, because total=False on this class

### 7.3 Nested TypedDict

In [ ]:
from typing import TypedDict

class Address(TypedDict):
    city: str
    country: str

class Customer(TypedDict):
    name: str
    address: Address

customer: Customer = {
    "name": "Ada",
    "address": {"city": "Lagos", "country": "Nigeria"},
}

### 7.4 API Responses, JSON Data, and Configuration Objects

In [ ]:
from typing import TypedDict, Literal

class ApiError(TypedDict):
    code: int
    message: str

class ApiResponse(TypedDict):
    status: Literal["success", "error"]
    data: dict[str, str] | None
    error: ApiError | None

def handle_response(response: ApiResponse) -> None:
    if response["status"] == "error" and response["error"] is not None:
        print(f"Error {response['error']['code']}: {response['error']['message']}")

### 7.5 TypedDict vs Dataclasses vs Ordinary Dictionaries

| Aspect | dict[str, X] | TypedDict | dataclass |
| --- | --- | --- | --- |
| Keys/fields known ahead of time | No - any string key allowed | Yes - fixed, named keys | Yes - fixed, named fields |
| Value types | All the same type X | Each field its own type | Each field its own type |
| Access syntax | obj["key"] | obj["key"] (still a dict!) | obj.field (attribute access) |
| Runtime type | dict | dict | A real class instance |
| Good for | Truly uniform, dynamic-key maps | JSON/API payloads, existing dict-based code | Internal domain objects, business logic |

## Part 7 (continued) - NamedTuple

### 7.6 Typed Tuple Structures

NamedTuple creates an immutable, tuple-based class with named, typed fields - it behaves like a lightweight, position-and-name-accessible record.

In [ ]:
from typing import NamedTuple

class Point(NamedTuple):
    x: float
    y: float

p = Point(1.0, 2.0)
print(p.x, p.y)     # 1.0 2.0 - named attribute access
print(p[0], p[1])   # 1.0 2.0 - still works positionally, like a normal tuple
x, y = p             # unpacking works too

### 7.7 Comparison With Ordinary Tuples and Dataclasses

| Aspect | Plain tuple | NamedTuple | dataclass |
| --- | --- | --- | --- |
| Field access | Only by position: t[0] | By name AND position | By name only: obj.field |
| Mutability | Immutable | Immutable | Mutable by default (can be frozen) |
| Memory footprint | Very low | Very low (same as tuple) | Slightly higher (a real object) |
| Best for | Quick, anonymous groupings | Small, fixed, immutable records | Richer domain objects with methods/logic |

### 7.8 Appropriate Use Cases for NamedTuple

- Simple, immutable coordinate/vector types (Point, RGBColor).

- Lightweight function return values that bundle a small, fixed number of related values (min_max(numbers) -> MinMax).

- Any place a plain tuple is currently used purely for its performance/immutability, but named access would improve readability.

## Part 7 (continued) - Dataclasses and Type Hints

### 7.9 Typed Fields and Default Values

In [ ]:
from dataclasses import dataclass

@dataclass
class Product:
    name: str
    price: float
    in_stock: bool = True   # default value

### 7.10 field() and default_factory

Mutable defaults (lists, dicts) cannot be used as plain default values in a dataclass (the same underlying issue as Python's classic mutable-default-argument pitfall). field(default_factory=...) solves this by calling a zero-argument function to produce a fresh object for every new instance.

In [ ]:
from dataclasses import dataclass, field

@dataclass
class ShoppingCart:
    owner: str
    items: list[str] = field(default_factory=list)   # a NEW empty list per instance

cart1 = ShoppingCart("Ada")
cart2 = ShoppingCart("Bo")
cart1.items.append("widget")
print(cart2.items)   # [] - correctly independent, thanks to default_factory

### 7.11 Optional Fields and Nested Dataclasses

In [ ]:
from dataclasses import dataclass

@dataclass
class Address:
    city: str
    country: str

@dataclass
class Customer:
    name: str
    address: Address
    phone: str | None = None

### 7.12 Frozen Dataclasses

frozen=True makes every field immutable after construction, raising a FrozenInstanceError on any later assignment attempt - useful for value objects that should never change once created, and for making instances hashable.

In [ ]:
from dataclasses import dataclass

@dataclass(frozen=True)
class Coordinates:
    latitude: float
    longitude: float

location = Coordinates(6.5244, 3.3792)
location.latitude = 0.0   # raises dataclasses.FrozenInstanceError

### 7.13 Dataclasses vs Dictionaries and vs Regular Classes

| Aspect | Plain dict | Regular class (manual __init__) | @dataclass |
| --- | --- | --- | --- |
| Boilerplate for __init__/repr/eq | None needed | You write it all by hand | Auto-generated from field annotations |
| Attribute access | obj["key"] | obj.field | obj.field |
| Static type checking | Weak (dict[str, X] at best) | Full, if annotated | Full, and enforced structurally |
| Best for | Ad hoc, dynamic data | Complex custom behavior | Structured records with light-to-moderate logic |

### Part 7 Common Mistakes and Best Practices

- Mistake: using a mutable default value directly in a dataclass field (field: list[str] = []) instead of field(default_factory=list) - this raises a ValueError at class-definition time specifically to prevent the classic shared-mutable-default bug.

- Mistake: reaching for a raw dict[str, Any] to represent a well-known JSON shape, when a TypedDict would give real field-name and type checking.

- Mistake: choosing NamedTuple for a record that actually needs mutability - NamedTuple instances cannot be modified after creation.

- Best practice: use TypedDict specifically when the value must remain a real dict at runtime (e.g. for JSON serialization); use a dataclass when you want a proper object with attribute access, methods, and optional immutability.

## Part 8 - ClassVar, Final, and Self

### 8.1 ClassVar: Instance Attributes vs Class Attributes

ClassVar tells a type checker that an annotated attribute belongs to the class itself, shared across every instance, and should never be set as a per-instance attribute. This distinction exists in plain Python already (see the earlier OOP handbook), but ClassVar makes it explicit and checkable.

In [ ]:
from typing import ClassVar
from dataclasses import dataclass

@dataclass
class Employee:
    name: str
    salary: float
    company_name: ClassVar[str] = "Acme Corp"   # shared by ALL Employee instances

e1 = Employee("Ada", 90000)
e2 = Employee("Bo", 85000)
print(e1.company_name, e2.company_name)   # Acme Corp, Acme Corp - same value

> **Why This Matters for Dataclasses:** A dataclass treats every plain annotated field as an instance field it must accept in __init__. ClassVar is the signal that tells @dataclass to SKIP that field entirely when generating __init__ - without ClassVar, company_name would incorrectly become a required constructor parameter.

### 8.2 Practical ClassVar Example: A Shared Counter

In [ ]:
from typing import ClassVar

class Order:
    total_orders: ClassVar[int] = 0

    def __init__(self, amount: float) -> None:
        self.amount = amount
        Order.total_orders += 1   # modifies the shared class-level counter

Order(50.0)
Order(75.0)
print(Order.total_orders)   # 2

## Part 8 (continued) - Final

### 8.3 Final: Preventing Reassignment at the Static-Analysis Level

Final marks a name (a variable, a class attribute, or an entire class/method) as one that a static type checker should flag if anyone ever tries to reassign or override it. Like every other annotation in this handbook, this is a static-analysis-time guarantee only - Python itself does not stop the reassignment at runtime.

In [ ]:
from typing import Final

MAX_CONNECTIONS: Final[int] = 100
MAX_CONNECTIONS = 200   # Type checker error: cannot assign to a Final name

# Python's interpreter, on its own, runs this line just fine:
print(MAX_CONNECTIONS)   # 200 - the reassignment DID happen at runtime

> **Static-Only Guarantee:** Final is enforced exclusively by the type checker you run (mypy, pyright). If you never run one, Final provides zero actual protection - it is documentation-as-a-contract, checked only when you invoke the check.

### 8.4 Final Class Attributes

In [ ]:
from typing import Final

class Config:
    API_VERSION: Final[str] = "v2"

### 8.5 Preventing Subclassing and Overriding

In [ ]:
from typing import final

@final
class ImmutablePoint:
    ...

class ExtendedPoint(ImmutablePoint):   # Type checker error: cannot subclass a @final class
    ...

class Base:
    @final
    def core_algorithm(self) -> int:
        return 42

class Sub(Base):
    def core_algorithm(self) -> int:   # Type checker error: cannot override a @final method
        return 0

Note the capitalization distinction: Final (capital F) is used as a type annotation for variables and attributes; final (lowercase) is the decorator form used on classes and methods to forbid subclassing or overriding.

## Part 8 (continued) - Self

### 8.6 What Self Typing Means (typing.Self, Python 3.11+)

typing.Self is a special annotation meaning 'whatever concrete class this method was actually called on' - crucially, this correctly follows through subclasses, which a hardcoded class name in the annotation cannot do. Do not confuse this typing.Self with the conventional self parameter name covered in the earlier Keywords handbook (Part 5) - they are two entirely different things that merely share a name.

In [ ]:
from typing import Self

class Builder:
    def __init__(self) -> None:
        self.parts: list[str] = []

    def add(self, part: str) -> Self:
        self.parts.append(part)
        return self   # returning 'self' the OBJECT; 'Self' the ANNOTATION describes its type

class AdvancedBuilder(Builder):
    def add_extra(self, extra: str) -> Self:
        self.parts.append(extra)
        return self

result = AdvancedBuilder().add("base").add_extra("extra")
# Thanks to 'Self', the type checker correctly knows 'result' is an
# AdvancedBuilder (not merely a Builder), so .add_extra() chains safely.

### 8.7 Fluent Interfaces and Builder-Style APIs

Self is precisely the tool that makes method-chaining ('fluent') APIs type-safe across inheritance. Before Self existed (pre-3.11), the only workaround was a TypeVar bound to the class itself, which was far more verbose.

In [ ]:
# Legacy workaround, pre-3.11:
from typing import TypeVar

TBuilder = TypeVar("TBuilder", bound="Builder")

class Builder:
    def add(self: TBuilder, part: str) -> TBuilder:
        ...
        return self

### 8.8 Self in Inheritance

Because Self always refers to 'the actual runtime class of the object', a base class method annotated -> Self remains correctly typed for every subclass automatically, without needing to override the annotation in each subclass - this is the core practical benefit over hardcoding the base class's own name as the return type.

### Part 8 Common Mistakes and Best Practices

- Mistake: forgetting ClassVar on a dataclass field that is genuinely meant to be shared, causing @dataclass to wrongly treat it as a required per-instance constructor argument.

- Mistake: assuming Final actually prevents reassignment at runtime - it does not, unless a type checker is run and its errors are treated as build failures.

- Mistake: annotating a fluent method's return type with the literal base class name instead of Self, silently breaking chaining accuracy for every subclass.

- Best practice: use Final for genuine constants and Self for any method whose whole purpose is to return the calling instance for chaining.

## Part 9 - Type Narrowing, TypeGuard, Forward References, Recursive Types, and Inference

### 9.1 How Type Checkers Narrow Types

Narrowing is the process by which a type checker follows the control flow of your code and concludes that, within a specific branch, a value's type is more specific than its declared annotation. Several everyday constructs trigger narrowing automatically.

#### isinstance()

In [ ]:
def process(value: int | str) -> str:
    if isinstance(value, int):
        return str(value * 2)   # here, 'value' is narrowed to int
    return value.upper()        # here, 'value' is narrowed to str

#### issubclass()

In [ ]:
def describe(cls: type) -> str:
    if issubclass(cls, Exception):
        return "This is an exception type"
    return "Some other type"

#### None Checks

In [ ]:
def greet(name: str | None) -> str:
    if name is not None:
        return name.upper()   # narrowed to plain str here
    return "Hello, stranger"

#### type()

type(x) is SomeClass also narrows, though isinstance() is generally preferred because it correctly accounts for subclasses, whereas type() only matches the exact class.

#### assert

In [ ]:
def get_length(value: str | None) -> int:
    assert value is not None, "value must not be None here"
    return len(value)   # narrowed to str after the assert

> **Cross-reference:** As covered in the earlier Keywords handbook (Part 6), assert statements can be stripped out entirely when Python runs with the -O flag. Relying on assert purely for narrowing is fine for internal invariants, but it is not a substitute for genuine input validation (Part 24) in production-facing code paths.

### 9.2 Why Narrowing Matters

Narrowing is what allows precise, Optional-safe, union-safe code to also be concise: instead of casting or re-annotating manually at every step, the type checker follows your ordinary if/isinstance/assert logic and automatically knows exactly which type is valid at each point in the code - matching how a careful human reader would reason about the same code.

## Part 9 (continued) - TypeGuard and TypeIs

### 9.3 User-Defined Type Guards

Sometimes the logic needed to narrow a type is more complex than a single isinstance() call - for example, checking that every element of a list matches a type. TypeGuard lets you write a custom function that a type checker will trust as a narrowing check, as long as it returns bool.

In [ ]:
from typing import TypeGuard

def is_str_list(values: list[object]) -> TypeGuard[list[str]]:
    return all(isinstance(v, str) for v in values)

def process(values: list[object]) -> None:
    if is_str_list(values):
        # type checker now treats 'values' as list[str] in this branch
        print(", ".join(values))

Line by line: -> TypeGuard[list[str]] tells the checker 'if this function returns True, trust that the argument was actually a list[str]', even though the parameter itself was only declared list[object]. The actual runtime check (all(isinstance(...))) must genuinely justify that promise - TypeGuard does not verify your logic is correct, it only wires your verified result into the type system.

### 9.4 TypeIs (Python 3.13+)

TypeIs is a refinement of TypeGuard, standardized in PEP 742, that narrows in BOTH branches (the true branch AND the false branch), whereas TypeGuard only narrows the true branch. Prefer TypeIs on 3.13+ when your check is a straightforward 'is this actually type X' test.

In [ ]:
from typing import TypeIs

def is_str(value: object) -> TypeIs[str]:
    return isinstance(value, str)

def handle(value: str | int) -> None:
    if is_str(value):
        print(value.upper())   # narrowed to str
    else:
        print(value + 1)       # TypeIs ALSO narrows the else-branch to int

### 9.5 Common TypeGuard Mistakes

- Writing a TypeGuard function whose body does not actually verify what the return type claims - the type checker trusts you completely and cannot catch a dishonest guard.

- Using TypeGuard when a plain isinstance() check would already narrow correctly - reach for TypeGuard only when the check genuinely cannot be a single isinstance/issubclass call.

- Forgetting that on pre-3.13 Python, TypeIs is unavailable, and TypeGuard's narrower (true-branch-only) behavior is what you get instead.

## Part 9 (continued) - Forward References and Recursive Types

### 9.6 Referencing Classes Before Definition

A forward reference is a type name written as a string, used when the referenced class has not been defined yet at the point the annotation appears - most commonly when a class references its own type (a linked list Node whose next: Node points to another Node).

In [ ]:
class Node:
    def __init__(self, value: int, next: "Node | None" = None) -> None:
        self.value = value
        self.next = next   # 'Node' isn't fully defined yet at this exact line,
                           # so the annotation above uses a quoted string

### 9.7 Modern Annotation Behavior: from __future__ import annotations

Adding from __future__ import annotations at the top of a module (PEP 563) makes every annotation in that file automatically treated as a string at runtime, meaning forward references no longer need manual quoting anywhere in that file. This has been the default in some form across recent PEPs, but as of the current stable release it is still an opt-in import, not the automatic default behavior.

In [ ]:
from __future__ import annotations

class Node:
    def __init__(self, value: int, next: Node | None = None) -> None:
        # No quotes needed around 'Node' here - the future import
        # makes all annotations lazily evaluated as strings.
        self.value = value
        self.next = next

### 9.8 Recursive Types

A recursive type refers to itself, directly or indirectly, in its own definition - essential for modeling trees, nested JSON, file systems, and linked data structures.

#### Nested JSON

In [ ]:
from typing import Union

type JsonValue = Union[str, int, float, bool, None, list["JsonValue"], dict[str, "JsonValue"]]

def process_json(data: JsonValue) -> None:
    ...

#### Tree Structures

In [ ]:
from dataclasses import dataclass, field

@dataclass
class TreeNode:
    value: int
    children: list["TreeNode"] = field(default_factory=list)

#### File-System Structures

In [ ]:
from typing import TypedDict

class FileEntry(TypedDict):
    name: str
    type: str   # "file" or "directory"
    children: list["FileEntry"]   # empty list for plain files

#### Linked Structures

In [ ]:
class ListNode:
    def __init__(self, value: int, next: "ListNode | None" = None) -> None:
        self.value = value
        self.next = next

## Part 9 (continued) - Type Inference

### 9.9 What Type Inference Means

Type inference is a static type checker's ability to determine a value's type automatically from context, without an explicit annotation. Python's type checkers infer types constantly: from literal values, from a function's declared return type, and from the results of already-typed operations.

In [ ]:
count = 5           # inferred as int, no annotation needed
name = "Ada"        # inferred as str
prices = [9.99, 4.5]   # inferred as list[float]

def get_name() -> str:
    return "Ada"

result = get_name()   # inferred as str, from the function's return annotation

### 9.10 When Explicit Annotations Are Still Useful

- The initial value gives no useful clue, most commonly an empty collection: items = [] infers list[Any] (or list[Unknown], depending on the checker) - items: list[str] = [] is far more useful.

- A function signature (parameters and return type) should almost always be explicitly annotated - inference works well for local variables, but a function's public interface is documentation other people (and other files) depend on.

- A value's type is deliberately being widened (e.g., you want a variable typed as a Protocol or base class even though you are currently assigning a specific subclass instance).

### 9.11 Avoiding Unnecessary Annotations

In [ ]:
# Unnecessary - the checker already infers this correctly:
count: int = 5

# Fine, and idiomatic:
count = 5

# Necessary - inference alone cannot determine the intended element type:
results: list[str] = []

> **Style Guidance:** Over-annotating every single local variable, even when inference is perfectly clear, adds visual noise without adding safety. Reserve explicit local annotations for the genuinely ambiguous cases (Part 41 covers this as a common mistake in more depth).

## Part 10 - Static Type Checkers, and Type Checking vs Runtime Validation

### 10.1 How Static Type Checking Works

A static type checker reads your source code without running it, builds an internal model of every variable, parameter, and return value's declared or inferred type, and then walks through your code checking that every operation is consistent with those types - a function call matches the parameter types, a returned value matches the declared return type, an attribute access exists on the inferred type, and so on. It reports every inconsistency it finds as an error or warning, entirely independent of whether the code would actually crash if run.

### 10.2 Mypy

Mypy is the original, reference-style Python static type checker, maintained under the Python organization itself. It is mature, highly configurable, and the checker most other tools (including this handbook's terminology) are described relative to.

#### Installing and Running Mypy

```bash
pip install mypy

mypy my_script.py
mypy my_project/          # check an entire directory/package
```

#### Reading Mypy Errors

In [ ]:
# my_script.py
def greet(name: str) -> str:
    return "Hello, " + name

greet(42)

```text
$ mypy my_script.py
my_script.py:4: error: Argument 1 to "greet" has incompatible type "int"; expected "str"  [arg-type]
Found 1 error in 1 file (checked 1 source file)
```

The bracketed code at the end of the error ([arg-type]) is a Mypy error code, which can be used to selectively silence or configure that specific category of check.

### 10.3 Pyright and Pylance

Pyright is a fast, standalone static type checker built by Microsoft, usable from the command line exactly like Mypy. Pylance is the VS Code extension that packages Pyright together with rich IDE features (autocomplete, inline error squiggles, hover-based type information), making Pyright the type checker most VS Code users are running without necessarily realizing it by name.

```bash
npm install -g pyright   # Pyright ships as an npm package
pyright my_script.py
```

| Tool | Distribution | Typical Use |
| --- | --- | --- |
| Mypy | pip install mypy | CI/CD pipelines, the most configurable option, cross-editor |
| Pyright | npm install -g pyright / pip install pyright | Fast CLI checks, also usable in CI |
| Pylance | VS Code extension (bundles Pyright) | Real-time in-editor checking and autocomplete |

### 10.4 Ruff's Type-Related Capabilities

Ruff is primarily an extremely fast linter and formatter, not a full static type checker - it does not perform the same deep type inference as Mypy or Pyright. However, Ruff does catch a number of typing-related style and correctness issues (unused typing imports, some outdated typing patterns, certain annotation-syntax errors) as part of its broader rule set, and is commonly run alongside a real type checker rather than instead of one.

### 10.5 Configuring a Type Checker

```toml
# pyproject.toml
[tool.mypy]
python_version = "3.12"
warn_return_any = true
disallow_untyped_defs = false   # relaxed - allows gradual adoption
exclude = ["tests/fixtures/"]
```

### 10.6 Strict Mode

Strict mode enables the full set of a checker's available checks at once - no implicit Any, no untyped function definitions allowed, no unchecked None access, and more. It is the recommended target for new projects, but is usually far too aggressive to turn on immediately for a large existing codebase.

```bash
mypy --strict my_project/

# Equivalent, in pyproject.toml:
[tool.mypy]
strict = true
```

### 10.7 Gradually Typing an Existing Project

- Start with a lenient configuration (disallow_untyped_defs = false) so existing unannotated code does not immediately produce hundreds of errors.

- Annotate new code fully and strictly from day one, even while the rest of the codebase remains loose.

- Use mypy's per-module strictness overrides to tighten one directory or package at a time as it gets annotated.

- Track error count over time as a metric, and ratchet the configuration stricter only once the current count reaches zero.

- Add the type checker to CI (Part 40) as soon as the baseline is clean, so no new untyped/incorrect code slips back in.

```toml
# pyproject.toml - per-module overrides during a gradual migration
[[tool.mypy.overrides]]
module = "legacy_module.*"
ignore_errors = true

[[tool.mypy.overrides]]
module = "new_module.*"
strict = true
```

## Part 10 (continued) - Type Checking vs Runtime Validation

### 10.8 The Distinction, Stated Plainly

Type hints do not automatically validate runtime input. This has already been stated in Part 1 and is repeated here deliberately, because it is precisely the confusion that causes real production bugs: developers sometimes believe that annotating a FastAPI endpoint parameter as int somehow protects the function from ever receiving a non-integer value at runtime, all on its own. It does not - only a library that actually reads those annotations and performs validation at runtime (like Pydantic, or FastAPI built on top of it) provides that protection, and it does so through its own separate validation machinery, not through Python's core type system.

### 10.9 Static Analysis vs Runtime Validation vs Data Validation vs Serialization

| Concept | When It Happens | What It Actually Does |
| --- | --- | --- |
| Static analysis | Before running the program (via a checker) | Verifies your CODE is internally consistent with its own annotations |
| Runtime validation | While the program runs, on real incoming data | Checks actual VALUES against rules, raising errors for bad data |
| Data validation | Same as runtime validation | A broader term - includes business rules, not just types |
| Serialization / deserialization | Converting between Python objects and formats like JSON | Often paired with validation (e.g. Pydantic does both at once) |

### 10.10 A Concrete Illustration

In [ ]:
def create_user(age: int) -> None:
    print(f"User age: {age}")

create_user("twenty-five")   # Runs WITHOUT any error at the interpreter level!
# Output: User age: twenty-five
# The annotation 'age: int' was never consulted at runtime.

A static type checker run against this file would flag the call immediately, before it ever executes. But if this function is, say, a request handler receiving raw data from a network request (which arrives as untyped, unchecked strings/JSON regardless of any annotation), only genuine runtime validation protects it in production.

### 10.11 When Pydantic or Similar Validation Is Appropriate

In [ ]:
from pydantic import BaseModel

class User(BaseModel):
    name: str
    age: int

User(name="Ada", age="twenty-five")
# pydantic.ValidationError: 1 validation error for User
# age
#   Input should be a valid integer, unable to parse string as an integer

Pydantic reuses the exact same annotation syntax you have been learning throughout this handbook, but it reads those annotations at runtime and actively enforces them - converting compatible values, and raising a real, catchable exception for incompatible ones. This is the bridge between 'type hints as documentation for a checker' and 'type hints as an active runtime guarantee', and it is why Pydantic has become the standard foundation for typed data models across FastAPI, LLM tool-calling schemas, and configuration systems.

> **Practical Rule of Thumb:** Reach for Pydantic (or a similar runtime-validation library) at every trust boundary: incoming HTTP requests, CLI arguments, environment variables, file/config parsing, and any data crossing from an external system into your typed Python code. Plain type hints alone, checked only statically, are not enough at these boundaries.

## Part 11 - Type Hints Across APIs, Data Analytics, Data Science, ML, and AI Engineering

This part applies everything covered so far to five domains where typed Python has become the professional norm. Each section is deliberately practical, using realistic shapes rather than toy examples.

### 11.1 Type Hints With APIs

#### Request and Response Objects

In [ ]:
from dataclasses import dataclass
from typing import TypedDict, Literal

@dataclass
class CreateUserRequest:
    name: str
    email: str
    age: int | None = None

class CreateUserResponse(TypedDict):
    id: int
    name: str
    email: str

#### Error Responses

In [ ]:
class ErrorResponse(TypedDict):
    status: Literal["error"]
    code: int
    message: str

def handle_error(response: ErrorResponse) -> None:
    print(f"[{response['code']}] {response['message']}")

#### A Typed API Client

In [ ]:
import requests
from typing import TypedDict

class WeatherData(TypedDict):
    temperature_c: float
    condition: str

class WeatherClient:
    def __init__(self, base_url: str, api_key: str) -> None:
        self.base_url = base_url
        self.api_key = api_key

    def get_current(self, city: str) -> WeatherData:
        response = requests.get(
            f"{self.base_url}/current",
            params={"city": city, "key": self.api_key},
        )
        response.raise_for_status()
        return response.json()   # NOTE: returns 'Any' from .json() at runtime -
                                 # the TypedDict return annotation is a STATIC promise,
                                 # not a runtime guarantee (see Part 10).

#### Configuration and HTTP Service Setup

In [ ]:
from dataclasses import dataclass

@dataclass(frozen=True)
class ServiceConfig:
    host: str
    port: int
    timeout_seconds: float = 30.0

### 11.2 Type Hints With Data Analytics (Pandas and NumPy)

Pandas and NumPy have improved, but still imperfect, typing support. DataFrame and Series are generally typed loosely (often close to Any internally for column contents), because a DataFrame's columns and their types are a runtime property, not something Python's static type system tracks column-by-column out of the box. Typing here mostly protects the functions AROUND your data (their inputs, outputs, and business logic), rather than fully verifying every column's dtype.

In [ ]:
import pandas as pd

def load_sales_data(path: str) -> pd.DataFrame:
    return pd.read_csv(path)

def clean_sales_data(df: pd.DataFrame) -> pd.DataFrame:
    df = df.dropna(subset=["amount"])
    df = df[df["amount"] > 0]
    return df

def total_revenue(df: pd.DataFrame) -> float:
    return float(df["amount"].sum())

#### Dataset Schemas

A common professional pattern is to describe an expected DataFrame's shape with a TypedDict or dataclass documenting the row structure, even though the DataFrame itself is not checked column-by-column - this keeps the intended schema explicit and reviewable, and pairs well with runtime validation libraries built for DataFrames (such as Pandera), which are the DataFrame-specific equivalent of Pydantic.

In [ ]:
from typing import TypedDict

class SalesRow(TypedDict):
    date: str
    region: str
    amount: float

# Documents intent; a library like Pandera would enforce it at runtime.
def process_sales_row(row: SalesRow) -> float:
    return row["amount"] * 1.075   # apply tax

#### ETL Pipeline Typing

In [ ]:
import pandas as pd

def extract(path: str) -> pd.DataFrame:
    return pd.read_csv(path)

def transform(df: pd.DataFrame) -> pd.DataFrame:
    df["total"] = df["price"] * df["quantity"]
    return df

def load(df: pd.DataFrame, destination: str) -> None:
    df.to_parquet(destination)

def run_pipeline(source: str, destination: str) -> None:
    df = extract(source)
    df = transform(df)
    load(df, destination)

> **NumPy Specifics:** NumPy provides numpy.typing.NDArray for annotating array shapes with a specific dtype, e.g. NDArray[np.float64]. Precise shape-checking (rows/columns as part of the type) is still an active area of the ecosystem, not something the standard type system verifies directly.

In [ ]:
import numpy as np
from numpy.typing import NDArray

def normalize(values: NDArray[np.float64]) -> NDArray[np.float64]:
    return (values - values.mean()) / values.std()

### 11.3 Type Hints With Data Science

#### Feature Engineering Functions

In [ ]:
import pandas as pd

def add_age_bucket(df: pd.DataFrame) -> pd.DataFrame:
    df["age_bucket"] = pd.cut(df["age"], bins=[0, 18, 35, 60, 120],
                              labels=["minor", "young_adult", "adult", "senior"])
    return df

#### Statistical Functions

In [ ]:
def z_score(value: float, mean: float, std_dev: float) -> float:
    return (value - mean) / std_dev

#### Data Preprocessing and Dataset Loaders

In [ ]:
from dataclasses import dataclass
import pandas as pd

@dataclass
class Dataset:
    features: pd.DataFrame
    labels: pd.Series

def load_dataset(path: str, label_column: str) -> Dataset:
    df = pd.read_csv(path)
    labels = df.pop(label_column)
    return Dataset(features=df, labels=labels)

#### Model Evaluation

In [ ]:
from typing import TypedDict

class EvaluationMetrics(TypedDict):
    accuracy: float
    precision: float
    recall: float
    f1_score: float

def evaluate(y_true: list[int], y_pred: list[int]) -> EvaluationMetrics:
    ...   # compute real metrics here

### 11.4 Type Hints With Machine Learning

#### Typed Preprocessing and Feature Vectors

In [ ]:
import numpy as np
from numpy.typing import NDArray

def scale_features(X: NDArray[np.float64]) -> NDArray[np.float64]:
    return (X - X.mean(axis=0)) / X.std(axis=0)

#### Typed Model Interfaces via Protocol

In [ ]:
from typing import Protocol
import numpy as np
from numpy.typing import NDArray

class SupportsFitPredict(Protocol):
    def fit(self, X: NDArray[np.float64], y: NDArray[np.float64]) -> None: ...
    def predict(self, X: NDArray[np.float64]) -> NDArray[np.float64]: ...

def train_and_evaluate(
    model: SupportsFitPredict,
    X_train: NDArray[np.float64],
    y_train: NDArray[np.float64],
    X_test: NDArray[np.float64],
) -> NDArray[np.float64]:
    model.fit(X_train, y_train)
    return model.predict(X_test)

# Works with scikit-learn estimators, custom models, or test doubles -
# any object with matching fit/predict methods, via structural typing.

#### Generic Model Interfaces and Pipeline Components

In [ ]:
from typing import TypeVar, Generic

TInput = TypeVar("TInput")
TOutput = TypeVar("TOutput")

class PipelineStep(Generic[TInput, TOutput]):
    def run(self, data: TInput) -> TOutput:
        raise NotImplementedError

class Tokenize(PipelineStep[str, list[str]]):
    def run(self, data: str) -> list[str]:
        return data.split()

### 11.5 Type Hints in AI Engineering

#### Prompt Objects and Model Configuration

In [ ]:
from dataclasses import dataclass
from typing import Literal

@dataclass
class ModelConfig:
    model_name: str
    temperature: float = 0.7
    max_tokens: int = 1024

@dataclass
class ChatMessage:
    role: Literal["system", "user", "assistant"]
    content: str

#### Structured Outputs and Tool/Function Calling

In [ ]:
from typing import TypedDict, Literal

class ToolParameter(TypedDict):
    name: str
    type: Literal["string", "number", "boolean"]
    description: str

class ToolDefinition(TypedDict):
    name: str
    description: str
    parameters: list[ToolParameter]

def get_weather_tool() -> ToolDefinition:
    return {
        "name": "get_weather",
        "description": "Get the current weather for a city",
        "parameters": [
            {"name": "city", "type": "string", "description": "City name"},
        ],
    }

#### Retrieval Pipelines, Embeddings, and Vector Search Results

In [ ]:
from dataclasses import dataclass

type Embedding = list[float]

@dataclass
class Document:
    id: str
    text: str
    embedding: Embedding | None = None

@dataclass
class SearchResult:
    document: Document
    score: float

def top_k_results(results: list[SearchResult], k: int) -> list[SearchResult]:
    return sorted(results, key=lambda r: r.score, reverse=True)[:k]

#### Document-Processing Pipelines and Agent Components

In [ ]:
from typing import Protocol

class DocumentLoader(Protocol):
    def load(self, path: str) -> str: ...

class TextChunker(Protocol):
    def chunk(self, text: str, size: int) -> list[str]: ...

class Agent:
    def __init__(self, loader: DocumentLoader, chunker: TextChunker) -> None:
        self.loader = loader
        self.chunker = chunker

    def ingest(self, path: str, chunk_size: int = 500) -> list[str]:
        text = self.loader.load(path)
        return self.chunker.chunk(text, chunk_size)

> **Design Note:** Notice how Protocol (Part 6) lets Agent accept any loader/chunker implementation - a local file loader, an S3 loader, a PDF-specific loader - without Agent depending on a concrete base class at all. This structural, dependency-injection-friendly style is extremely common in professional AI-engineering codebases.

### Part 11 Common Mistakes and Best Practices

- Mistake: expecting pd.DataFrame or NDArray annotations to verify column names or array shapes - they do not, by themselves; pair them with a schema-validation library where that guarantee genuinely matters.

- Mistake: typing every ML pipeline step as Any because 'the data is complex', instead of using Protocol or Generic to preserve at least the input/output relationship.

- Best practice: model API and LLM payloads with TypedDict or dataclasses rather than raw dict[str, Any], so the field-level structure is checkable and self-documenting.

- Best practice: use Protocol for anything that plugs into a pipeline (loaders, models, chunkers) so implementations stay swappable and testable.

## Part 12 - Typing Decorators: ParamSpec, Concatenate, and Annotated

### 12.1 The Challenge of Typing Decorators

A decorator wraps a function and typically returns a new function. The core typing challenge is preserving the ORIGINAL function's exact parameter list and return type through the wrapper, so that callers of the decorated function still get full, accurate type checking - rather than the wrapper collapsing everything down to Callable[..., Any].

### 12.2 A Basic, Loosely Typed Decorator

In [ ]:
from typing import Callable, Any
import functools

def logged(func: Callable[..., Any]) -> Callable[..., Any]:
    @functools.wraps(func)
    def wrapper(*args: Any, **kwargs: Any) -> Any:
        print(f"Calling {func.__name__}")
        return func(*args, **kwargs)
    return wrapper

This works, but every decorated function's signature is flattened to (..., Any) as far as the type checker is concerned - callers lose all real parameter and return-type checking, which defeats much of the purpose of typing the rest of the codebase.

### 12.3 functools.wraps

functools.wraps(func) copies over func's __name__, __doc__, and other metadata onto the wrapper, which is essential for debugging and introspection - but on its own it does not restore static type information; that requires ParamSpec, below.

### 12.4 ParamSpec: Why It Exists

ParamSpec (PEP 612, Python 3.10+) is a special kind of type variable that captures an ENTIRE parameter list - all positional and keyword parameters together, including their names and defaults - as a single reusable unit, rather than one type at a time like a normal TypeVar.

In [ ]:
from typing import Callable, TypeVar
from typing import ParamSpec
import functools

P = ParamSpec("P")
R = TypeVar("R")

def logged(func: Callable[P, R]) -> Callable[P, R]:
    @functools.wraps(func)
    def wrapper(*args: P.args, **kwargs: P.kwargs) -> R:
        print(f"Calling {func.__name__}")
        return func(*args, **kwargs)
    return wrapper

@logged
def add(a: int, b: int) -> int:
    return a + b

add(2, 3)      # type checker fully validates: two ints, returns int
add("2", 3)    # type checker error - caught, unlike the basic decorator above

Line by line: Callable[P, R] means 'a callable whose entire parameter list is P, and whose return type is R.' Inside wrapper, *args: P.args and **kwargs: P.kwargs re-declare that this wrapper accepts exactly func's original parameter list, and -> R declares it returns exactly what func returns. The combination of P and R is what fully preserves the original signature through the decorator.

### 12.5 Preserving Arbitrary Parameter Lists in Higher-Order Functions

In [ ]:
from typing import Callable, ParamSpec, TypeVar
import time

P = ParamSpec("P")
R = TypeVar("R")

def timed(func: Callable[P, R]) -> Callable[P, R]:
    def wrapper(*args: P.args, **kwargs: P.kwargs) -> R:
        start = time.perf_counter()
        result = func(*args, **kwargs)
        print(f"{func.__name__} took {time.perf_counter() - start:.4f}s")
        return result
    return wrapper

## Part 12 (continued) - Concatenate

### 12.6 Adding Parameters to a Callable's Signature

Concatenate (also PEP 612) lets you describe a callable that has one or more EXTRA leading parameters beyond a ParamSpec's captured list - the classic case being a decorator that injects an additional argument (like a logger, a request context, or self) before the original parameters.

In [ ]:
from typing import Callable, Concatenate, ParamSpec, TypeVar

P = ParamSpec("P")
R = TypeVar("R")

def with_logger(func: Callable[Concatenate[str, P], R]) -> Callable[P, R]:
    def wrapper(*args: P.args, **kwargs: P.kwargs) -> R:
        logger_name = "default"
        return func(logger_name, *args, **kwargs)
    return wrapper

@with_logger
def process(logger_name: str, value: int) -> int:
    print(f"[{logger_name}] processing {value}")
    return value * 2

process(5)   # 'logger_name' is injected automatically; caller only supplies 'value'

Line by line: Callable[Concatenate[str, P], R] describes the ORIGINAL function's full signature - a leading str parameter (logger_name), followed by whatever P represents. The decorator's returned wrapper is typed Callable[P, R] - notice logger_name has been consumed/removed from what the CALLER needs to supply, which is exactly what the real runtime behavior does.

### 12.7 Combining Concatenate With ParamSpec in Advanced Decorators

In [ ]:
from typing import Callable, Concatenate, ParamSpec, TypeVar

P = ParamSpec("P")
R = TypeVar("R")

class Cache:
    def __init__(self) -> None:
        self.store: dict[object, object] = {}

def with_cache(
    func: Callable[Concatenate[Cache, P], R]
) -> Callable[Concatenate[Cache, P], R]:
    def wrapper(cache: Cache, *args: P.args, **kwargs: P.kwargs) -> R:
        key = (args, tuple(sorted(kwargs.items())))
        if key not in cache.store:
            cache.store[key] = func(cache, *args, **kwargs)
        return cache.store[key]   # type: ignore[return-value]
    return wrapper

## Part 12 (continued) - Annotated

### 12.8 Adding Metadata to Types With Annotated

Annotated[Type, metadata...] attaches arbitrary extra metadata to a type annotation, without changing what the underlying type actually is to a type checker (a checker still sees just Type). Tools that specifically know how to look for that metadata (Pydantic, FastAPI, and others) can read it and use it to drive validation, documentation, or serialization - tools that don't recognize it simply ignore the extra metadata.

In [ ]:
from typing import Annotated

UserId = Annotated[int, "Must be a positive integer"]

def get_user(user_id: UserId) -> None:
    ...

# To a plain type checker, UserId behaves exactly like int.
# The string metadata is inert unless something reads it deliberately.

### 12.9 Annotated in Validation and API Frameworks

In [ ]:
from typing import Annotated
from pydantic import Field, BaseModel

class CreateUserRequest(BaseModel):
    age: Annotated[int, Field(gt=0, lt=120)]
    username: Annotated[str, Field(min_length=3, max_length=20)]

# Pydantic reads the Field(...) metadata inside Annotated and actively
# enforces those constraints at runtime when a CreateUserRequest is built.

In [ ]:
from typing import Annotated
from fastapi import FastAPI, Query

app = FastAPI()

@app.get("/items/")
def read_items(
    q: Annotated[str | None, Query(max_length=50)] = None,
) -> dict[str, str | None]:
    return {"q": q}

### 12.10 Practical Data Model Example

In [ ]:
from typing import Annotated
from dataclasses import dataclass

PositiveFloat = Annotated[float, "must be > 0"]

@dataclass
class Product:
    name: str
    price: PositiveFloat   # documents intent; still needs real validation to enforce it

### Part 12 Common Mistakes and Best Practices

- Mistake: writing a decorator typed as Callable[..., Any] -> Callable[..., Any] purely out of habit, silently erasing type safety for every function it decorates.

- Mistake: forgetting functools.wraps entirely, which breaks introspection (help(), debuggers, some frameworks) even when ParamSpec correctly preserves the STATIC type.

- Mistake: assuming Annotated metadata does something on its own - it is inert unless a specific library is written to read and act on it.

- Best practice: reach for ParamSpec + TypeVar as the default, professional way to type any general-purpose decorator meant for reuse across many differently-signed functions.

## Part 13 - Async Typing, Exceptions, NewType, and Overload

### 13.1 Coroutine and Awaitable

An async def function, when called, produces a coroutine object. Coroutine[YieldType, SendType, ReturnType] is the precise generic type for it, though in most everyday code Awaitable[ReturnType] (a broader, simpler protocol satisfied by anything usable with await) is what you will annotate a parameter or variable with.

In [ ]:
from typing import Awaitable
import asyncio

async def fetch_data(url: str) -> str:
    await asyncio.sleep(0.1)
    return f"data from {url}"

def schedule(coro: Awaitable[str]) -> None:
    asyncio.run(coro)   # type checker knows: awaiting 'coro' yields a str

Note that fetch_data itself is simply annotated with a normal return type (-> str), exactly as if it were synchronous - the async def keyword already tells both Python and the type checker that calling it produces an awaitable; you do not additionally wrap the return annotation in Coroutine yourself for ordinary async functions.

### 13.2 AsyncIterator and AsyncIterable

In [ ]:
from typing import AsyncIterator

async def stream_lines(path: str) -> AsyncIterator[str]:
    # (a real implementation would use an async file library)
    for line in ["a", "b", "c"]:
        yield line

async def consume() -> None:
    async for line in stream_lines("data.txt"):
        print(line)

AsyncIterable[T] describes anything usable in an async for loop (requires __aiter__); AsyncIterator[T] additionally supports __anext__ directly. An async generator function (one using yield inside an async def, as above) is the most common way to produce an AsyncIterator.

### 13.3 AsyncGenerator

In [ ]:
from typing import AsyncGenerator

async def countdown(start: int) -> AsyncGenerator[int, None]:
    current = start
    while current > 0:
        yield current
        current -= 1

AsyncGenerator[YieldType, SendType] mirrors the synchronous Generator[YieldType, SendType, ReturnType] shape from the standard typing module, minus the explicit return-type parameter, since async generators cannot use a value-carrying return statement.

### 13.4 Typed Async API Clients

In [ ]:
from typing import TypedDict
import aiohttp

class UserData(TypedDict):
    id: int
    name: str

class AsyncApiClient:
    def __init__(self, base_url: str) -> None:
        self.base_url = base_url

    async def get_user(self, user_id: int) -> UserData:
        async with aiohttp.ClientSession() as session:
            async with session.get(f"{self.base_url}/users/{user_id}") as resp:
                return await resp.json()

### 13.5 Async AI/LLM Applications

In [ ]:
from typing import AsyncIterator
from dataclasses import dataclass

@dataclass
class StreamChunk:
    text: str
    done: bool

async def stream_completion(prompt: str) -> AsyncIterator[StreamChunk]:
    for word in ["Hello", ", ", "world", "!"]:
        yield StreamChunk(text=word, done=False)
    yield StreamChunk(text="", done=True)

async def print_stream(prompt: str) -> None:
    async for chunk in stream_completion(prompt):
        print(chunk.text, end="")

## Part 13 (continued) - Type Hints and Exceptions

### 13.6 Return Types Involving Errors

Python functions communicate failure in two very different ways - raising an exception, or returning a sentinel/union value (like None, or a Result-style object) - and the annotation style should honestly reflect which one a given function actually does.

In [ ]:
# Style 1: signals failure via None (caller must check)
def find_user(user_id: int) -> "User | None":
    ...

# Style 2: signals failure via a raised exception (caller must try/except)
def get_user(user_id: int) -> "User":
    """Raises UserNotFoundError if the user does not exist."""
    ...

### 13.7 NoReturn: Functions That Always Raise

NoReturn documents that a function never returns normally at all - every path through it either raises or otherwise never completes (an infinite loop with no break, for instance). It is most common for dedicated error-raising helper functions and functions like sys.exit().

In [ ]:
from typing import NoReturn

def fail(message: str) -> NoReturn:
    raise RuntimeError(message)

def process(value: int | None) -> int:
    if value is None:
        fail("value must not be None")
    return value * 2   # type checker knows this line is only reached if value is not None,
                       # BECAUSE fail() is marked NoReturn - narrowing benefits from it too

### 13.8 Exception-Aware API Design and Avoiding Misleading Annotations

- Do not annotate a function's return type as X when it can also raise and never actually return in the failure case that callers will hit often - document raised exceptions clearly (a docstring, or a custom exception type) instead of pretending failure doesn't happen.

- Do not annotate a return type more narrowly than what is actually returned (e.g. -> int when None is sometimes returned) - this is a direct, checkable lie to both readers and the type checker.

- Prefer a specific custom exception class (Part 6 of the Keywords handbook) over a generic Exception, so callers can catch precisely what they intend to handle.

## Part 13 (continued) - NewType

### 13.9 Creating Semantically Distinct Types

NewType creates a distinct 'flavor' of an existing type, purely for the static type checker - at runtime, a NewType is just the underlying type with no wrapper object at all (it is a zero-cost identity function). Its entire purpose is to stop two values that happen to share a underlying type (both are int, both are str) but represent conceptually different things from being accidentally interchanged.

In [ ]:
from typing import NewType

UserId = NewType("UserId", int)
ProductId = NewType("ProductId", int)

def get_user(user_id: UserId) -> str:
    return f"User #{user_id}"

uid = UserId(42)
pid = ProductId(42)

get_user(uid)   # OK
get_user(pid)   # Type checker error: ProductId is not a UserId, even though both are int
get_user(42)    # Also a type checker error: a raw int is not a UserId either

Line by line: UserId = NewType("UserId", int) creates a callable, UserId(...), that a type checker treats as producing a genuinely distinct type from plain int (and from ProductId), even though at runtime UserId(42) simply evaluates to the int 42 with zero wrapping overhead.

### 13.10 Why NewType Prevents Accidental Mixing

Account numbers, user IDs, product IDs, and similar identifiers are extremely easy to accidentally swap when they are all just plain int or str - a function call like transfer_funds(destination_account, source_account) with the arguments reversed compiles and runs without complaint if both are plain int. Wrapping each in its own NewType turns that entire class of bug into a static type-checker error, catchable before the code ever runs.

In [ ]:
from typing import NewType

AccountId = NewType("AccountId", str)

def transfer_funds(source: AccountId, destination: AccountId, amount: float) -> None:
    ...

acc_a = AccountId("ACC-001")
acc_b = AccountId("ACC-002")
transfer_funds(acc_a, acc_b, 100.0)   # fine, but a checker cannot tell WHICH order is 'correct' -
                                       # NewType prevents mixing with unrelated types, not argument order

> **Precise Scope:** NewType prevents mixing DIFFERENT semantic types that share an underlying representation (UserId vs ProductId, both int). It does not, by itself, prevent swapping two parameters of the SAME NewType - for that, more descriptive keyword-only parameters or distinct dataclasses would be the stronger tool.

## Part 13 (continued) - overload

### 13.11 Function Overload Declarations

Python does not support true function overloading at the language level - defining the same function name twice simply replaces the first definition. typing.overload lets you DOCUMENT, for the type checker only, that a function has multiple valid call signatures with different, more precise return types depending on the input - while a single real implementation handles all of them at runtime.

In [ ]:
from typing import overload

@overload
def parse(value: str) -> int: ...
@overload
def parse(value: list[str]) -> list[int]: ...

def parse(value: str | list[str]) -> int | list[int]:
    if isinstance(value, str):
        return int(value)
    return [int(v) for v in value]

x = parse("42")          # type checker knows: x is int
y = parse(["1", "2"])    # type checker knows: y is list[int] (not the wider union)

Line by line: the two @overload-decorated stubs declare the precise input-to-output type pairs a caller can rely on; neither has a real body (... is a placeholder, never executed). The final, undecorated def parse(...) is the ONE real implementation, whose signature must be broad enough (the union) to legally cover every overload declared above it - callers, however, see only the precise overload that matches their specific argument type.

### 13.12 Static Overload Resolution vs Runtime Implementation

| Aspect | @overload stubs | Real implementation |
| --- | --- | --- |
| Executed at runtime? | No - body is always just ... | Yes - this is the only code that actually runs |
| Purpose | Tell the checker the precise per-input return type | Handle every case the stubs promised, at runtime |
| Count per function | Two or more | Exactly one, placed last, undecorated |

### Part 13 Common Mistakes and Best Practices

- Mistake: forgetting that async def already implies an awaitable return - double-wrapping the return annotation in Coroutine[...] unnecessarily for ordinary async functions.

- Mistake: annotating a function's return type dishonestly narrow when it can also return None or raise in common, expected cases.

- Mistake: using @overload but writing an implementation signature that does not actually cover every overloaded case - a type checker will flag this inconsistency.

- Best practice: reach for NewType specifically for identifiers and other 'same underlying type, different meaning' values that are easy to mix up by accident.

## Part 14 - Type Hints, OOP, and Design Patterns

### 14.1 Typed Constructors and Methods

In [ ]:
class BankAccount:
    def __init__(self, owner: str, balance: float = 0.0) -> None:
        self.owner = owner
        self.balance = balance

    def deposit(self, amount: float) -> None:
        self.balance += amount

    def withdraw(self, amount: float) -> bool:
        if amount > self.balance:
            return False
        self.balance -= amount
        return True

### 14.2 Typed Inheritance

In [ ]:
class Animal:
    def __init__(self, name: str) -> None:
        self.name = name

    def speak(self) -> str:
        raise NotImplementedError

class Dog(Animal):
    def speak(self) -> str:
        return f"{self.name} says Woof"

A subclass overriding a method should keep a compatible signature: the parameter types may only be widened (a subclass method may accept a broader type than the parent declared) and the return type may only be narrowed (a subclass method may promise a more specific return type) - this rule is called Liskov-substitutability, and static type checkers actively enforce it.

### 14.3 Generic Classes in OOP (Recap)

Part 5 already covered Generic[T] in depth. In OOP-heavy code, generics are most useful for reusable containers and base classes (repositories, collections, wrappers) that need to stay type-precise for whatever concrete entity they hold.

### 14.4 Protocol-Based Design vs Abstract Base Classes

Python also offers abc.ABC for nominal abstract base classes, which require explicit inheritance and can enforce that certain methods are overridden. Protocol (Part 6) achieves a similar 'contract' goal structurally, without requiring inheritance.

In [ ]:
from abc import ABC, abstractmethod

class Shape(ABC):
    @abstractmethod
    def area(self) -> float: ...

class Circle(Shape):
    def __init__(self, radius: float) -> None:
        self.radius = radius

    def area(self) -> float:
        return 3.14159 * self.radius ** 2

# Circle() alone (without an area() override) would raise TypeError at
# instantiation time - ABC enforces this at RUNTIME, unlike Protocol.

| Aspect | abc.ABC | typing.Protocol |
| --- | --- | --- |
| Requires inheritance | Yes | No |
| Enforcement | Runtime (TypeError on missing method) | Static only (type checker) |
| Can provide shared implementation | Yes (regular methods alongside abstract ones) | No (method bodies are placeholders) |
| Best for | A true, owned class hierarchy | Decoupled, structural interfaces (including third-party types) |

### 14.5 Polymorphism With Type Hints

In [ ]:
def total_area(shapes: list[Shape]) -> float:
    return sum(shape.area() for shape in shapes)

shapes: list[Shape] = [Circle(2.0), Circle(3.0)]
print(total_area(shapes))

## Part 14 (continued) - Type Hints and Design Patterns

### 14.6 Factory Pattern

In [ ]:
from typing import Literal

class Dog:
    def speak(self) -> str:
        return "Woof"

class Cat:
    def speak(self) -> str:
        return "Meow"

def create_animal(kind: Literal["dog", "cat"]) -> Dog | Cat:
    if kind == "dog":
        return Dog()
    return Cat()

### 14.7 Strategy Pattern

In [ ]:
from typing import Protocol

class DiscountStrategy(Protocol):
    def apply(self, price: float) -> float: ...

class NoDiscount:
    def apply(self, price: float) -> float:
        return price

class TenPercentOff:
    def apply(self, price: float) -> float:
        return price * 0.9

def checkout(price: float, strategy: DiscountStrategy) -> float:
    return strategy.apply(price)

### 14.8 Repository Pattern

In [ ]:
from typing import Protocol, TypeVar

T = TypeVar("T")

class Repository(Protocol[T]):
    def get(self, entity_id: int) -> T | None: ...
    def add(self, entity: T) -> int: ...

class InMemoryUserRepository:
    def __init__(self) -> None:
        self._data: dict[int, "User"] = {}
        self._next_id = 1

    def get(self, entity_id: int) -> "User | None":
        return self._data.get(entity_id)

    def add(self, entity: "User") -> int:
        self._data[self._next_id] = entity
        self._next_id += 1
        return self._next_id - 1

### 14.9 Adapter Pattern

In [ ]:
from typing import Protocol

class ModernLogger(Protocol):
    def log(self, message: str) -> None: ...

class LegacyLogger:
    def write_log_entry(self, text: str) -> None:
        print(f"[legacy] {text}")

class LegacyLoggerAdapter:
    def __init__(self, legacy: LegacyLogger) -> None:
        self._legacy = legacy

    def log(self, message: str) -> None:
        self._legacy.write_log_entry(message)

def run_with_logger(logger: ModernLogger) -> None:
    logger.log("starting up")

run_with_logger(LegacyLoggerAdapter(LegacyLogger()))

### 14.10 Dependency Injection

In [ ]:
from typing import Protocol

class EmailSender(Protocol):
    def send(self, to: str, subject: str, body: str) -> None: ...

class UserService:
    def __init__(self, email_sender: EmailSender) -> None:
        self._email_sender = email_sender   # injected dependency, typed structurally

    def register(self, email: str) -> None:
        self._email_sender.send(email, "Welcome!", "Thanks for signing up.")

### 14.11 Pipeline Architecture

In [ ]:
from typing import TypeVar, Generic, Callable

T = TypeVar("T")

class Pipeline(Generic[T]):
    def __init__(self, value: T) -> None:
        self.value = value

    def then(self, step: Callable[[T], T]) -> "Pipeline[T]":
        return Pipeline(step(self.value))

result = (
    Pipeline("  hello world  ")
    .then(str.strip)
    .then(str.upper)
    .value
)
print(result)   # HELLO WORLD

### Part 14 Common Mistakes and Best Practices

- Mistake: narrowing a parameter type when overriding a method (violating Liskov substitutability), which a type checker will correctly flag as unsafe.

- Mistake: choosing ABC when the real goal is a lightweight, decoupled interface across unrelated classes - Protocol is usually the better fit for pattern-based designs like Strategy and Adapter.

- Best practice: type dependency-injection constructor parameters as Protocol rather than a concrete class, so tests can supply lightweight fakes without inheritance.

- Best practice: keep pattern implementations (Factory, Strategy, Repository) fully annotated, since these are exactly the seams where a wrong type is most likely to slip through unnoticed.

## Part 15 - Professional Project Structure, Common Mistakes, cast(), and Best Practices

### 15.1 Introducing Type Hints Into a Real Project: Structure

```text
my_project/
|-- pyproject.toml
|-- src/
|   `-- my_project/
|       |-- __init__.py
|       |-- py.typed          <- marker file: this package ships type information
|       |-- models.py
|       |-- services.py
|       `-- api/
|           |-- __init__.py
|           `-- routes.py
|-- tests/
|   |-- test_models.py
|   `-- test_services.py
`-- .github/
    `-- workflows/
        `-- ci.yml
```

The py.typed marker file (PEP 561) is what tells OTHER projects that import yours that your package ships its own type annotations and can be type-checked against directly, rather than being treated as untyped by default.

### 15.2 pyproject.toml Type-Checker Configuration

```toml
[tool.mypy]
python_version = "3.12"
strict = true
exclude = ["tests/fixtures/"]

[tool.pyright]
include = ["src"]
typeCheckingMode = "strict"
pythonVersion = "3.12"
```

### 15.3 Source Code Organization for Typed Projects

- Keep pure data models (dataclasses, TypedDicts) separate from business logic, so type-checking pure models is fast and their shape is easy to review.

- Define shared type aliases and Protocols in a dedicated types.py (or a types/ package) once they are used across multiple modules.

- Avoid circular-import-driven forward references where possible by organizing modules so dependencies flow in one direction.

### 15.4 Tests and Type Checking

Test files should be type-checked too - a surprising number of real bugs (wrong fixture types, mismatched mock return values) are caught this way. Some teams intentionally relax strictness slightly for test code (e.g. allowing Any in fixtures) while keeping application code fully strict.

```toml
[[tool.mypy.overrides]]
module = "tests.*"
disallow_untyped_defs = false
```

### 15.5 CI/CD Integration

```yaml
# .github/workflows/ci.yml
name: CI
on: [push, pull_request]
jobs:
  type-check:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: "3.12"
      - run: pip install mypy
      - run: mypy src/
```

Making the type-check step a required, blocking check on pull requests is what turns type hints from 'nice documentation someone might read' into an actively enforced quality gate - the single highest-leverage step a team can take to get real, ongoing value from typing.

### 15.6 Documentation

Well-annotated function signatures reduce how much needs to be repeated in docstrings, but they do not eliminate docstrings entirely - annotations describe types; docstrings should still describe behavior, side effects, raised exceptions, and non-obvious constraints that types alone cannot express (see Part 13's discussion of exception-aware design).

## Part 15 (continued) - Common Type-Hinting Mistakes

### 15.7 The Full Mistake List

- Using Any everywhere as a default instead of investigating the real type - covered in depth in Part 4.

- Over-annotating simple, obviously-inferred local variables, adding noise without adding safety - covered in Part 9.

- Incorrect Optional usage: giving a parameter a None default while annotating it as a non-Optional type - covered in Part 3.

- Confusing runtime validation with static type checking, and believing annotations alone protect against bad input - covered in depth in Part 10.

- Incorrect generic syntax: writing tuple[int] when tuple[int, ...] was meant, or forgetting Generic[T] on a class that uses a bare TypeVar - covered in Parts 2 and 5.

- Ignoring type-checker errors, or blanket-suppressing them with # type: ignore instead of fixing or narrowly justifying each one.

- Using overly complicated nested generic types instead of a type alias or a proper TypedDict/dataclass - covered in Parts 2, 4, and 7.

- Making annotations harder to understand than the underlying code - the fastest sign of this is a signature spanning several lines of nested brackets for a genuinely simple function.

- Incorrectly typing decorators, collapsing every decorated function's signature to Callable[..., Any] - covered in Part 12.

- Misusing cast() to silence an error instead of fixing the actual type mismatch - covered next, in 15.9.

In [ ]:
# Ignoring errors, badly:
value = get_config()  # type: ignore

# Better: suppress only the specific, understood error code, with a reason:
value = get_config()  # type: ignore[assignment]  # legacy config returns Any; TODO: fix upstream

## Part 15 (continued) - cast()

### 15.8 What cast() Does

typing.cast(Type, value) tells the type checker 'trust me, treat this value as Type from this point forward' - and that is ALL it does. It performs zero runtime conversion, zero runtime checking, and has zero runtime effect beyond returning the exact same object you passed in, unchanged.

In [ ]:
from typing import cast, Any

def get_config_value(key: str) -> Any:
    config: dict[str, Any] = {"port": "8080"}   # imagine this loaded from JSON
    return config[key]

port = cast(int, get_config_value("port"))
print(port)          # "8080" - STILL A STRING at runtime!
print(port + 1)      # TypeError: can only concatenate str (not "int") to str

> **Critical Distinction:** cast() is a promise to the type checker, not a conversion function. The example above compiles cleanly under a type checker (which now believes port is an int) but crashes at runtime, because cast() never actually called int(...) on anything. This is precisely why cast() is considered dangerous when misused.

### 15.9 Appropriate Use Cases

- You have already verified the type through logic a checker cannot follow (a regex match you know always succeeds given prior validation), and want to silence a spurious, false-positive error.

- Interfacing with a loosely-typed third-party library whose stub returns Any or a wider type than you know is actually returned in your specific usage.

- Narrowing after your own custom runtime check that a TypeGuard (Part 9) would be more work to write than the situation warrants (though TypeGuard is generally the better long-term choice).

### 15.10 Dangerous Use Cases

- Using cast() to make a type-checker error disappear without actually verifying, anywhere in the code, that the value truly is that type - this is functionally the same risk as using Any, but harder to spot on review because it looks precise.

- Casting the result of external, untrusted input (a network response, user input) without genuine runtime validation alongside it.

In [ ]:
# The RIGHT way to fix the port example above - actually convert it:
port = int(get_config_value("port"))   # a genuine runtime conversion, not a cast
print(port + 1)   # 8081 - works correctly

## Part 15 (continued) - Professional Best Practices

### 15.11 When to Annotate

- Always annotate function and method signatures (parameters and return types) - this is the highest-value, lowest-cost place to add types.

- Always annotate class attributes, especially dataclass fields, since these often define your data model's public contract.

- Annotate local variables only when inference genuinely cannot determine the intended type (Part 9).

- Always annotate module-level constants, especially when paired with Final.

### 15.12 How Strict a Project Should Be

| Project Stage | Recommended Strictness |
| --- | --- |
| Brand-new project | Strict mode from day one - there is no legacy debt to work around |
| Existing project, adding types for the first time | Lenient config, tightened gradually per module (Part 10.7) |
| Mature, fully-typed project | Strict mode enforced in CI, treated as a hard merge requirement |

### 15.13 Naming Conventions, Readability, and Maintainability

- Give type aliases and Protocols clear, descriptive, domain-specific names (UserId, JsonValue, SupportsClose) rather than generic ones (T1, MyType).

- Keep a single TypeVar named T when it is truly generic and unconstrained; give constrained/bounded TypeVars a name that hints at the restriction.

- Favor several small, focused types over one enormous union or an overly generic catch-all - readability of the annotation matters as much as readability of the code it describes.

### 15.14 Gradual Typing as an Ongoing Practice, Not a One-Time Project

Even a fully-typed, strict-mode codebase needs ongoing discipline: new third-party dependencies may lack stubs, new contributors may be unfamiliar with the project's typing conventions, and new language features (Part 4's type statement, Part 9's TypeIs) will periodically offer better ways to express existing types. Treat the type-checker configuration itself as a piece of code that occasionally needs updating and revisiting, not a settings file to write once and forget.

## Part 16 - Real-World Projects (1 of 2)

Each project below follows the same structure: problem statement, requirements, architecture, type definitions, a complete runnable implementation, a line-by-line-style explanation of the key typing decisions, a type-checking strategy, a testing strategy, and possible improvements.

### Project 1 - Typed Student Management System

#### Problem Statement

Build a small system to register students, record grades, and compute each student's average and letter grade, with full static type safety.

#### Requirements

- Model a Student with a name and a list of numeric grades.

- Compute an average grade and a letter grade (A/B/C/F).

- Support looking students up by ID, returning None if not found.

#### Architecture

A single StudentRegistry class holds a dict[int, Student]. Student is a dataclass. Letter grades are modeled with Literal for a closed, checkable set of values.

#### Type Definitions and Implementation

In [ ]:
from dataclasses import dataclass, field
from typing import Literal

LetterGrade = Literal["A", "B", "C", "F"]

@dataclass
class Student:
    name: str
    grades: list[float] = field(default_factory=list)

    def add_grade(self, grade: float) -> None:
        self.grades.append(grade)

    def average(self) -> float | None:
        if not self.grades:
            return None
        return sum(self.grades) / len(self.grades)

    def letter_grade(self) -> LetterGrade | None:
        avg = self.average()
        if avg is None:
            return None
        if avg >= 90:
            return "A"
        if avg >= 80:
            return "B"
        if avg >= 70:
            return "C"
        return "F"

class StudentRegistry:
    def __init__(self) -> None:
        self._students: dict[int, Student] = {}
        self._next_id = 1

    def register(self, name: str) -> int:
        student_id = self._next_id
        self._students[student_id] = Student(name)
        self._next_id += 1
        return student_id

    def find(self, student_id: int) -> Student | None:
        return self._students.get(student_id)

registry = StudentRegistry()
sid = registry.register("Ada")
student = registry.find(sid)
if student is not None:
    student.add_grade(95)
    student.add_grade(88)
    print(student.letter_grade())   # A

#### Explanation

Every method that can meaningfully fail to produce a value (average, letter_grade, find) returns an Optional type rather than a sentinel like -1 or an exception, which keeps the 'no data yet' case impossible to ignore at every call site, thanks to type-checker-enforced narrowing (Part 9).

#### Type-Checking Strategy

Run mypy --strict against this module; strict mode will flag any call site that forgets to check for None before calling a method on the result of find() or average().

#### Testing Strategy

Unit tests should cover: a student with no grades (average/letter_grade return None), a typical student, and find() on a nonexistent ID.

#### Improvements

- Replace the plain int student ID with a NewType (Part 13) to prevent mixing it up with other numeric IDs elsewhere in a larger system.

- Add a TypedDict-based export format for serializing a Student to JSON for an API layer (Part 25).

### Project 2 - Typed Sales Analytics Pipeline

#### Problem Statement

Process a list of raw sales records into cleaned data and a summary report, with every transformation stage fully typed.

#### Requirements

- Accept raw sales rows as TypedDicts (as they might arrive from a CSV/JSON source).

- Filter out invalid rows (non-positive amounts).

- Compute total revenue and revenue by region.

#### Architecture

A simple functional pipeline: extract -> clean -> summarize, each stage typed with a TypedDict input/output shape.

#### Type Definitions and Implementation

In [ ]:
from typing import TypedDict

class RawSale(TypedDict):
    region: str
    amount: float

class SalesSummary(TypedDict):
    total_revenue: float
    revenue_by_region: dict[str, float]

def clean_sales(rows: list[RawSale]) -> list[RawSale]:
    return [row for row in rows if row["amount"] > 0]

def summarize(rows: list[RawSale]) -> SalesSummary:
    by_region: dict[str, float] = {}
    for row in rows:
        by_region[row["region"]] = by_region.get(row["region"], 0.0) + row["amount"]
    return {
        "total_revenue": sum(row["amount"] for row in rows),
        "revenue_by_region": by_region,
    }

raw_rows: list[RawSale] = [
    {"region": "West", "amount": 120.0},
    {"region": "East", "amount": -5.0},   # invalid, will be filtered
    {"region": "West", "amount": 40.0},
]
clean_rows = clean_sales(raw_rows)
report = summarize(clean_rows)
print(report)

#### Explanation

Because RawSale is a TypedDict, both clean_sales and summarize get full field-name and value-type checking on every dict access (row["amount"]) - a typo like row["ammount"] would be caught statically, unlike with a plain dict[str, float].

#### Type-Checking Strategy

Enable strict TypedDict checking (the default in both mypy and pyright); verify no plain dict literal is passed where a RawSale is expected without matching every required key.

#### Testing Strategy

Test clean_sales with a mix of valid/invalid rows, and summarize with an empty list, a single region, and multiple regions.

#### Improvements

- Swap the TypedDicts for Pandera-validated pandas DataFrames if the pipeline needs to scale to large datasets (Part 11.2).

- Add a Protocol-based SalesSource so the pipeline can pull from a CSV, a database, or an API interchangeably.

### Project 3 - Typed Financial Transaction Processor

#### Problem Statement

Process deposits, withdrawals, and transfers between accounts safely, using NewType to prevent account-ID mixups and a Result-style return to make failure explicit.

#### Requirements

- Model distinct AccountId and TransactionId types.

- Support deposit, withdraw, and transfer operations.

- Represent success/failure without relying on exceptions for ordinary business failures (insufficient funds).

#### Architecture

A TransactionResult TypedDict models both outcomes uniformly; a Ledger class holds balances keyed by the NewType AccountId.

#### Type Definitions and Implementation

In [ ]:
from typing import NewType, TypedDict, Literal

AccountId = NewType("AccountId", str)

class TransactionResult(TypedDict):
    status: Literal["success", "failure"]
    message: str

class Ledger:
    def __init__(self) -> None:
        self._balances: dict[AccountId, float] = {}

    def open_account(self, account_id: AccountId, initial: float = 0.0) -> None:
        self._balances[account_id] = initial

    def deposit(self, account_id: AccountId, amount: float) -> TransactionResult:
        self._balances[account_id] = self._balances.get(account_id, 0.0) + amount
        return {"status": "success", "message": f"Deposited {amount}"}

    def withdraw(self, account_id: AccountId, amount: float) -> TransactionResult:
        balance = self._balances.get(account_id, 0.0)
        if amount > balance:
            return {"status": "failure", "message": "Insufficient funds"}
        self._balances[account_id] = balance - amount
        return {"status": "success", "message": f"Withdrew {amount}"}

    def transfer(self, source: AccountId, destination: AccountId, amount: float) -> TransactionResult:
        result = self.withdraw(source, amount)
        if result["status"] == "failure":
            return result
        return self.deposit(destination, amount)

ledger = Ledger()
acc_a = AccountId("ACC-A")
acc_b = AccountId("ACC-B")
ledger.open_account(acc_a, 100.0)
ledger.open_account(acc_b, 0.0)
print(ledger.transfer(acc_a, acc_b, 40.0))   # {'status': 'success', ...}

#### Explanation

AccountId as a NewType prevents accidentally passing a raw, unrelated string (or a different kind of ID entirely) where an account identifier is required. TransactionResult communicates failure as ordinary, checkable data rather than a raised exception, since insufficient funds is an expected business outcome, not a programming error.

#### Type-Checking Strategy

Confirm the checker rejects transfer("ACC-A", acc_b, 40.0) - a raw string, not wrapped in AccountId(...) - demonstrating NewType's protection in practice.

#### Testing Strategy

Test insufficient-funds withdrawal, a successful transfer, and a transfer that fails partway (withdrawal succeeds conceptually but deposit destination is invalid) to confirm no funds are lost silently.

#### Improvements

- Add a proper Decimal-based amount type instead of float to avoid floating-point rounding issues in real financial code.

- Introduce a TransactionId NewType and a transaction log for full auditability.

### Project 4 - Typed ETL Pipeline

#### Problem Statement

Build a generic Extract-Transform-Load pipeline where each stage is swappable and independently type-checked, using Protocol and Generic together.

#### Requirements

- Define generic Extractor, Transformer, and Loader Protocols.

- Compose them into a single run_pipeline function that works for any matching set of components.

#### Architecture

Three small Protocols, each generic over the relevant input/output type, composed by a single orchestration function.

#### Type Definitions and Implementation

In [ ]:
from typing import Protocol, TypeVar

TRaw = TypeVar("TRaw")
TClean = TypeVar("TClean")

class Extractor(Protocol[TRaw]):
    def extract(self) -> list[TRaw]: ...

class Transformer(Protocol[TRaw, TClean]):
    def transform(self, item: TRaw) -> TClean: ...

class Loader(Protocol[TClean]):
    def load(self, items: list[TClean]) -> None: ...

def run_pipeline(
    extractor: Extractor[TRaw],
    transformer: Transformer[TRaw, TClean],
    loader: Loader[TClean],
) -> None:
    raw_items = extractor.extract()
    clean_items = [transformer.transform(item) for item in raw_items]
    loader.load(clean_items)

class CsvExtractor:
    def extract(self) -> list[str]:
        return ["1,Ada", "2,Bo"]

class CsvRowTransformer:
    def transform(self, item: str) -> dict[str, str]:
        id_, name = item.split(",")
        return {"id": id_, "name": name}

class PrintLoader:
    def load(self, items: list[dict[str, str]]) -> None:
        for item in items:
            print(item)

run_pipeline(CsvExtractor(), CsvRowTransformer(), PrintLoader())

Note: Protocol[TRaw] generic-Protocol syntax requires Generic to also be inherited on Python versions prior to fuller PEP 695 support; check your target Python version's exact requirements before relying on this pattern.

#### Explanation

None of CsvExtractor, CsvRowTransformer, or PrintLoader inherits from the Protocol classes - they satisfy them purely structurally (Part 6). run_pipeline is fully generic: swapping in a JsonExtractor and a different Transformer works without modifying run_pipeline at all, and the type checker verifies every stage's output type matches the next stage's expected input type.

#### Type-Checking Strategy

Deliberately mismatch a Transformer's output type against the Loader's expected input type in a test file, and confirm the checker reports the mismatch - this validates the Protocol chain is actually being enforced.

#### Testing Strategy

Unit test each component (Extractor, Transformer, Loader) independently, plus one integration test running the full pipeline end to end with fakes.

#### Improvements

- Add error handling per stage, using a Result-style TypedDict as in Project 3.

- Parameterize run_pipeline to accept multiple transformers chained in sequence.

## Part 16 - Real-World Projects (2 of 2)

### Project 5 - Typed Machine-Learning Preprocessing Pipeline

#### Problem Statement

Build a chainable, typed preprocessing pipeline for tabular ML features, ensuring each step's input/output array shape assumptions are documented and checkable.

#### Requirements

- Support a sequence of typed transformation steps (scaling, imputing missing values).

- Preserve NDArray typing throughout the chain.

#### Architecture

A Protocol-based Transform step interface, plus a Pipeline class that applies a list of steps in order.

#### Type Definitions and Implementation

In [ ]:
from typing import Protocol
import numpy as np
from numpy.typing import NDArray

FloatArray = NDArray[np.float64]

class Transform(Protocol):
    def apply(self, X: FloatArray) -> FloatArray: ...

class MeanImputer:
    def apply(self, X: FloatArray) -> FloatArray:
        col_means = np.nanmean(X, axis=0)
        indices = np.where(np.isnan(X))
        X[indices] = np.take(col_means, indices[1])
        return X

class StandardScaler:
    def apply(self, X: FloatArray) -> FloatArray:
        return (X - X.mean(axis=0)) / X.std(axis=0)

class Pipeline:
    def __init__(self, steps: list[Transform]) -> None:
        self.steps = steps

    def run(self, X: FloatArray) -> FloatArray:
        for step in self.steps:
            X = step.apply(X)
        return X

pipeline = Pipeline([MeanImputer(), StandardScaler()])
data: FloatArray = np.array([[1.0, np.nan], [3.0, 4.0]])
processed = pipeline.run(data)

#### Explanation

FloatArray is a type alias (Part 4) that keeps every signature in the file readable while still communicating the exact expected dtype. Every Transform implementation is structurally interchangeable, so new preprocessing steps can be added without modifying Pipeline.

#### Type-Checking Strategy

Run mypy with numpy's plugin/stub support enabled; confirm that passing an int-dtype array where float64 is expected is flagged (numpy's typing precision varies by version - verify behavior against your installed numpy/mypy versions).

#### Testing Strategy

Test each Transform independently against small, hand-computed arrays, then test Pipeline.run with a short, deterministic sequence.

#### Improvements

- Add a fit/transform split (separate 'learn parameters from training data' from 'apply to any data') to match scikit-learn conventions.

- Wrap the pipeline output in a small dataclass carrying both the array and the column names it corresponds to.

### Project 6 - Typed REST API Client

#### Problem Statement

Build a small, typed client for a hypothetical task-management REST API, modeling requests, responses, and errors explicitly.

#### Requirements

- Typed methods for listing, creating, and completing tasks.

- A distinct, typed error path for HTTP failures.

#### Architecture

A dataclass-based Task model, TypedDict wire-format types for raw JSON, and a client class translating between them.

#### Type Definitions and Implementation

In [ ]:
from dataclasses import dataclass
from typing import TypedDict
import requests

class TaskJson(TypedDict):
    id: int
    title: str
    completed: bool

@dataclass
class Task:
    id: int
    title: str
    completed: bool

    @classmethod
    def from_json(cls, data: TaskJson) -> "Task":
        return cls(id=data["id"], title=data["title"], completed=data["completed"])

class ApiError(Exception):
    def __init__(self, status_code: int, message: str) -> None:
        super().__init__(message)
        self.status_code = status_code

class TaskApiClient:
    def __init__(self, base_url: str) -> None:
        self.base_url = base_url

    def list_tasks(self) -> list[Task]:
        response = requests.get(f"{self.base_url}/tasks")
        if not response.ok:
            raise ApiError(response.status_code, "Failed to list tasks")
        raw: list[TaskJson] = response.json()
        return [Task.from_json(item) for item in raw]

    def create_task(self, title: str) -> Task:
        response = requests.post(f"{self.base_url}/tasks", json={"title": title})
        if not response.ok:
            raise ApiError(response.status_code, "Failed to create task")
        return Task.from_json(response.json())

    def complete_task(self, task_id: int) -> Task:
        response = requests.patch(f"{self.base_url}/tasks/{task_id}", json={"completed": True})
        if not response.ok:
            raise ApiError(response.status_code, "Failed to complete task")
        return Task.from_json(response.json())

#### Explanation

TaskJson models the wire format exactly as the API returns it; Task is the clean internal domain object the rest of the application actually uses. from_json is the single, explicit translation point between them, which is where a validation library like Pydantic would be introduced if untrusted-input safety became a requirement (Part 10.11).

#### Type-Checking Strategy

Confirm response.json() (typed Any by the requests stub) is only ever consumed through Task.from_json, never accessed with raw, unchecked dict keys elsewhere in the client.

#### Testing Strategy

Mock requests.get/post/patch, returning representative JSON payloads, and test both the success and ApiError paths for each method.

#### Improvements

- Replace manual TaskJson/from_json translation with a Pydantic model for actual runtime validation of the API's response shape.

- Add retry-with-backoff logic, typed via a small RetryPolicy dataclass.

### Project 7 - Typed NLP Processing Pipeline

#### Problem Statement

Build a small text-processing pipeline (tokenize, remove stopwords, count word frequencies) with typed intermediate representations at every stage.

#### Requirements

- Tokenize raw text into a list of words.

- Filter out a configurable stopword set.

- Produce a word-frequency count.

#### Architecture

A sequence of small, purely-functional, fully typed transformation functions, composed by one orchestration function.

#### Type Definitions and Implementation

In [ ]:
from collections import Counter

def tokenize(text: str) -> list[str]:
    return text.lower().split()

def remove_stopwords(tokens: list[str], stopwords: set[str]) -> list[str]:
    return [token for token in tokens if token not in stopwords]

def word_frequencies(tokens: list[str]) -> dict[str, int]:
    return dict(Counter(tokens))

def process_text(text: str, stopwords: set[str]) -> dict[str, int]:
    tokens = tokenize(text)
    filtered = remove_stopwords(tokens, stopwords)
    return word_frequencies(filtered)

STOPWORDS = {"the", "a", "is", "and"}
result = process_text("the cat and the dog is happy", STOPWORDS)
print(result)   # {'cat': 1, 'dog': 1, 'happy': 1}

#### Explanation

Every function's input and output type exactly matches the next function's expected input, so process_text reads as a straight-line, type-verified composition - a mismatch introduced by an edit (e.g. word_frequencies accidentally returning a list instead of a dict) would be caught immediately by the checker at the process_text call site.

#### Type-Checking Strategy

Confirm that swapping the order of remove_stopwords and word_frequencies inside process_text produces a type error (since word_frequencies expects list[str], not dict[str, int]).

#### Testing Strategy

Test each function independently with edge cases: empty text, text that is entirely stopwords, and repeated words for frequency counting.

#### Improvements

- Replace the plain tokenizer with a real NLP library's tokenizer, wrapped behind a Protocol so it can be swapped without touching downstream stages.

- Add a TypedDict-based NlpResult carrying both the frequency table and metadata (token count, unique word count).

### Project 8 - Typed AI/LLM Application

#### Problem Statement

Build a small, typed 'agent' that accepts a user message, decides whether to call a tool, and returns a structured response - a simplified but realistic shape of a production LLM application.

#### Requirements

- Typed chat messages and typed tool definitions.

- A typed decision result: either a direct text reply, or a tool call to execute.

- A typed final response after any tool call is resolved.

#### Architecture

A closed Literal-tagged union (via TypedDict variants distinguished by a status/type field) models the model's decision; a small Agent class orchestrates the loop.

#### Type Definitions and Implementation

In [ ]:
from typing import TypedDict, Literal, Protocol
from dataclasses import dataclass

@dataclass
class ChatMessage:
    role: Literal["system", "user", "assistant"]
    content: str

class ToolCallDecision(TypedDict):
    kind: Literal["tool_call"]
    tool_name: str
    arguments: dict[str, str]

class DirectReplyDecision(TypedDict):
    kind: Literal["direct_reply"]
    text: str

Decision = ToolCallDecision | DirectReplyDecision

class LanguageModel(Protocol):
    def decide(self, messages: list[ChatMessage]) -> Decision: ...

class ToolRegistry:
    def __init__(self) -> None:
        self._tools: dict[str, "object"] = {}

    def register(self, name: str, func: "Protocol") -> None:  # simplified for brevity
        self._tools[name] = func

    def call(self, name: str, arguments: dict[str, str]) -> str:
        func = self._tools[name]
        return func(**arguments)   # type: ignore[operator]

class Agent:
    def __init__(self, model: LanguageModel, tools: ToolRegistry) -> None:
        self.model = model
        self.tools = tools

    def run(self, messages: list[ChatMessage]) -> str:
        decision = self.model.decide(messages)
        if decision["kind"] == "direct_reply":
            return decision["text"]
        # decision is narrowed to ToolCallDecision here
        return self.tools.call(decision["tool_name"], decision["arguments"])

The ToolRegistry.call method's use of type: ignore acknowledges a genuine typing gap around dynamically dispatched callables with heterogeneous signatures - a realistic situation worth annotating honestly rather than hiding behind an unjustified cast().

#### Explanation

Decision is a discriminated union: both TypedDict variants share a kind field with distinct Literal values, which is exactly the pattern that lets if decision["kind"] == "direct_reply": narrow decision precisely in each branch (Part 9) - mirroring how real LLM tool-calling APIs commonly tag their response variants.

#### Type-Checking Strategy

Verify that accessing decision["tool_name"] inside the direct_reply branch (before narrowing) is flagged as an error, confirming the discriminated union is actually providing branch-specific safety.

#### Testing Strategy

Use a fake LanguageModel that returns a fixed Decision, to test both the direct-reply and tool-call branches of Agent.run deterministically, without calling a real model.

#### Improvements

- Type ToolRegistry.call precisely per tool using overload (Part 13) once the set of tools is fixed and known ahead of time.

- Add Pydantic models for validating the LLM's raw JSON output against the Decision shape at runtime, since an actual model response is untrusted external input (Part 10.11).

## Part 18 - 60 Interview Questions With Answers

Organized into six sections of ten questions each: Beginner, Intermediate, Advanced, Senior Python Developer, Data Scientist, and AI Engineer. Each includes the answer and, where useful, a short supporting example.

### Beginner (Q1-Q10)

**Q1. What is a type hint, and does Python enforce it at runtime?**

A type hint is optional syntax documenting the intended type of a variable, parameter, or return value. Python's interpreter never enforces it at runtime - only an external static type checker (or a runtime-validation library like Pydantic) actually acts on it.

**Q2. What is the difference between List and list in annotations?**

List (capitalized) is the legacy typing.List, required before Python 3.9 to parameterize a list type. list (lowercase, built-in) can be parameterized directly since Python 3.9 (PEP 585) and is the modern, preferred style.

**Q3. What does Optional[str] mean?**

It means the value is either a str or None - exactly equivalent to str | None (the modern syntax, 3.10+) or Union[str, None].

**Q4. What is the output of this code, and why?**

Because annotations are not enforced at runtime, add("2", "3") runs the + operator on two strings, which concatenates them rather than raising an error.

In [ ]:
def add(a: int, b: int) -> int:
    return a + b

print(add("2", "3"))
# Output: 23   (string concatenation, NOT integer addition)

**Q5. What is the correct way to annotate a function with no meaningful return value?**

Annotate its return type as -> None, explicitly documenting that callers should not use the return value.

**Q6. What is the difference between annotating a variable and assigning a value to it?**

score: int declares the intended type without creating the variable; score: int = 0 both declares the type and creates the binding. Referencing score after only the bare annotation raises a NameError.

**Q7. How do you annotate a list containing only strings?**

names: list[str] on Python 3.9+, or names: List[str] with from typing import List on earlier versions.

**Q8. What does tuple[int, ...] mean, as opposed to tuple[int]?**

tuple[int, ...] means a tuple of any length where every element is an int. tuple[int] means a tuple of exactly one element, which is an int.

**Q9. Why should if value == None: generally be avoided in favor of if value is None:?**

None is a singleton (only one instance ever exists), so identity comparison (is) is the semantically correct and idiomatic check; == could theoretically be fooled by a custom __eq__ method on some other object.

**Q10. What tool would you run to actually check your type hints before running your program?**

A static type checker such as Mypy (mypy script.py) or Pyright (pyright script.py) - plain python script.py never validates annotations.

### Intermediate (Q11-Q20)

**Q11. What is the difference between Union[int, str] and Literal[1, 2, 3]?**

Union[int, str] allows any int or any str value. Literal[1, 2, 3] allows only those three exact integer values - it restricts to specific values, not an entire type.

**Q12. What does typing.Any actually do to static checking?**

It disables meaningful checking for that value entirely - Any is compatible with every type in both directions, and using it silences errors on attribute access, calls, and assignments involving that value.

**Q13. What will a strict type checker report for this code, and why?**

It reports an error, because the function signature declares -> int, but the actual return type when items is empty is None, not int - the annotation is dishonestly narrow.

In [ ]:
def first(items: list[int]) -> int:
    if not items:
        return None   # error: Incompatible return value type (got 'None', expected 'int')
    return items[0]

**Q14. What is a TypeVar, and why is it needed instead of Any?**

A TypeVar is a placeholder that preserves the relationship between an input type and an output type across a specific call, something Any cannot express since Any carries no information linking one usage to another.

**Q15. What is the difference between TypedDict and a dataclass?**

A TypedDict describes a plain dict with fixed keys and per-key types, and remains a real dict at runtime. A dataclass generates a genuine class with attribute access, __init__, and other dunder methods auto-generated from its fields.

**Q16. What does ClassVar indicate, and why does it matter for dataclasses?**

ClassVar marks an attribute as belonging to the class itself, not to each instance. In a dataclass specifically, it also tells @dataclass to exclude that field from the generated __init__ - without it, the field would incorrectly become a required constructor argument.

**Q17. What is structural typing, and how does Protocol implement it in Python?**

Structural typing means compatibility is determined by an object's shape (its methods/attributes) rather than its declared inheritance. Protocol lets you define a required shape that any class can satisfy without inheriting from the Protocol at all.

**Q18. What does Callable[[int, str], bool] describe?**

A callable taking exactly two parameters, an int followed by a str, and returning a bool.

**Q19. Why might Optional[int] = [] be a design smell in a dataclass field?**

It isn't Optional[int] specifically but a mutable default like list[str] = [] that is the smell - a plain mutable default is shared across every instance; field(default_factory=list) must be used instead.

**Q20. What is the practical difference between == and is when checking for None?**

== checks value equality (and could theoretically be overridden by a class's __eq__); is checks object identity. Since None is a true singleton, is None is both faster and more correct.

### Advanced (Q21-Q30)

**Q21. What problem does ParamSpec solve that a plain TypeVar cannot?**

A plain TypeVar captures a single type. ParamSpec captures an entire parameter list (names, positions, defaults) as one unit, which is exactly what a generic decorator needs to preserve the wrapped function's full original signature.

**Q22. What does Concatenate add on top of ParamSpec?**

Concatenate lets you describe one or more extra LEADING parameters in addition to a captured ParamSpec - the standard case being a decorator that injects an argument (like a resource or context object) before the original parameters.

**Q23. What is the difference between TypeGuard and TypeIs?**

TypeGuard only narrows the type in the branch where the guard function returns True. TypeIs (3.13+) narrows in BOTH the true and false branches, which is more precise for simple 'is this exactly type X' checks.

**Q24. What exactly does cast() do at runtime?**

Nothing. cast(Type, value) returns value completely unchanged - it performs no conversion and no check. It exists purely to tell the type checker to trust a specific type going forward.

**Q25. Why is this code type-safe but still crashes at runtime? Explain the bug.**

cast() never converts anything - it only affects the static type checker's belief about the value. The actual runtime value returned by get_raw_port() is still a str, so port + 1 fails with a TypeError despite the checker believing port is an int.

In [ ]:
from typing import cast

def get_raw_port() -> object:
    return "8080"

port = cast(int, get_raw_port())
print(port + 1)   # TypeError: can only concatenate str (not "int") to str

**Q26. What is a discriminated union, and how do you model one with TypedDict?**

A discriminated union is a set of alternative shapes that all share one common 'tag' field with distinct Literal values, letting a type checker narrow precisely once that tag is checked - modeled as a Union of several TypedDicts, each with its own Literal value for the shared tag field.

**Q27. Why does @overload require a final, undecorated implementation?**

The @overload-decorated stubs are never executed - they exist purely to declare precise per-input-type signatures for the checker. Exactly one real, undecorated implementation must exist to actually run at call time, and its signature must be broad enough to legally satisfy every declared overload.

**Q28. What is the difference between NewType and a type alias (TypeAlias / type statement)?**

A type alias (UserId: TypeAlias = int, or type UserId = int) is just another NAME for the same type - int and UserId are fully interchangeable to the checker. NewType creates a genuinely DISTINCT type for the checker (UserId is not interchangeable with plain int), while still being a zero-cost identity function at runtime.

**Q29. Why might a static type checker fail to fully validate a pandas DataFrame's columns?**

A DataFrame's columns and their dtypes are a runtime property of the actual data, not something the standard Python type system tracks per-column in its static type. pd.DataFrame is typed too loosely, by design, to catch a wrong-column-name bug - a dedicated schema-validation library (e.g. Pandera) is needed for that guarantee.

**Q30. What is the practical risk of overusing # type: ignore across a codebase?**

Each unqualified # type: ignore silences ALL errors on that line, including future, unrelated ones introduced by later edits - it accumulates hidden risk exactly like commented-out test assertions, and should be scoped to a specific error code with a short justification.

### Senior Python Developer (Q31-Q40)

**Q31. How would you introduce type hints into a 50,000-line legacy codebase without breaking anything?**

Start with a lenient checker configuration (no strict mode, disallow_untyped_defs off), add py.typed and CI integration early so the baseline never regresses, annotate new/touched code fully as you go, and tighten configuration module-by-module using per-module overrides until the whole codebase reaches strict mode.

**Q32. How do you decide between Mypy and Pyright for a team's CI pipeline?**

Both are valid; Mypy is generally more configurable and has the largest plugin ecosystem (useful for Django, SQLAlchemy-heavy projects), while Pyright is typically faster and is the engine behind Pylance, making it a natural fit for teams standardized on VS Code. Many teams run one in CI and rely on the other's IDE integration day-to-day.

**Q33. What is the Liskov substitution principle's implication for method overriding in a typed codebase?**

An overriding method's parameter types may only be widened (accept more than the parent promised) and its return type may only be narrowed (promise something more specific) relative to the parent - anything else breaks substitutability, and a type checker will flag it.

**Q34. When would you choose Protocol over abc.ABC for an internal interface?**

Choose Protocol when you want structural, inheritance-free compatibility (useful for third-party types, or to keep implementations decoupled and easily testable with lightweight fakes). Choose ABC when you also want runtime enforcement (TypeError on missing methods) and/or shared implementation code across subclasses.

**Q35. How do you type a function whose behavior legitimately differs by input type, with precise per-case return types?**

Use @overload to declare each precise input-to-output signature pair, followed by one real, undecorated implementation whose signature is a union broad enough to cover every overload.

**Q36. What is your strategy for typing third-party libraries with poor or missing type stubs?**

Check for a separate types- stub package (e.g. types-requests) first; if none exists, write a minimal local .pyi stub for just the parts you use, or isolate the untyped calls behind a small, fully-typed wrapper module so the untyped surface area stays contained.

**Q37. How does py.typed affect consumers of your published package?**

Its presence (an empty marker file, per PEP 561) tells type checkers that your package ships genuine inline type information and can be checked against directly; without it, checkers typically treat an installed third-party package as untyped by default, even if its source code has annotations.

**Q38. What tradeoffs come with enabling strict mode immediately on an existing large project?**

Strict mode will likely surface hundreds or thousands of pre-existing errors at once, which can be overwhelming and demoralizing, and risks the team disabling type checking altogether rather than working through the backlog - gradual, per-module tightening is usually the more sustainable path.

**Q39. How would you type a plugin system where third parties implement your interfaces?**

Define the required interface as a Protocol (or an ABC if shared base behavior and runtime enforcement are also needed), publish it as part of your public API, and accept plugin instances typed against that Protocol/ABC in your registration function.

**Q40. What is your review checklist for a pull request that adds new Any usage?**

Confirm the Any is genuinely unavoidable (truly untyped external data, or temporary migration debt with a tracked follow-up), check whether object or a Protocol would be a safer, still-checked alternative, and ensure the Any does not leak into a public function's return type where it would silently spread to every caller.

### Data Scientist (Q41-Q50)

**Q41. Why does typing a pandas-heavy data pipeline still add value, even without column-level checking?**

It still verifies the SHAPE of your code around the data - which functions accept a DataFrame vs a Series, which return None vs a DataFrame, and which parameters are genuinely optional - catching an entire class of 'wrong object type passed to the wrong stage' bugs even without full column awareness.

**Q42. How would you document an expected DataFrame schema in a typed codebase?**

Define a TypedDict (or a dataclass) describing the expected row/column structure as documentation and a code-review aid, and pair it with a runtime schema-validation library like Pandera for an actual enforced guarantee, since plain pd.DataFrame typing cannot verify columns on its own.

**Q43. What does numpy.typing.NDArray[np.float64] communicate, and what does it not guarantee?**

It communicates the expected element dtype (64-bit floats). It does not, on its own, guarantee or check the array's SHAPE (number of dimensions or size), which remains a runtime property.

**Q44. How would you type a feature-engineering function that adds a new column to a DataFrame?**

def add_feature(df: pd.DataFrame) -> pd.DataFrame: is standard - taking and returning a DataFrame documents that the function is part of a chainable transformation pipeline, consistent with Part 11.2 and Project 2's approach.

**Q45. Why might Any appear frequently in typed data-science code, and how do you keep it contained?**

Many DataFrame/Series operations return values close to Any internally because pandas' typing does not track dtypes column-by-column. Contain it by keeping DataFrame manipulation isolated in a small set of functions, with clearly typed inputs/outputs at the BOUNDARIES of your pipeline, even if internals are loosely typed.

**Q46. What is the benefit of a TypedDict-based row schema over a bare dict[str, Any] for CSV-derived data?**

TypedDict gives you per-key type checking and typo detection (row["amout"] would be flagged) at the point you access fields, whereas dict[str, Any] provides no such protection at all.

**Q47. How would you type a function that either returns computed statistics or None if the input dataset is empty?**

Annotate its return type as StatsResult | None (using Optional or the | syntax), and require every caller to check for None before accessing fields, exactly as with Project 1's average() method.

**Q48. What role does Protocol play in making a model-evaluation function reusable across different ML libraries?**

A Protocol like SupportsFitPredict (Part 11.4) lets an evaluation function accept any object with matching fit/predict methods, whether it's a scikit-learn estimator, a custom model, or a test double, without depending on any specific library's base class.

**Q49. Why is float typically the right annotation for statistical results, even though many individual computations return numpy scalar types?**

Python's numeric types and numpy's scalar types are close enough that most checkers and codebases treat this pragmatically as float for readability at the boundaries; teams needing stricter guarantees can use numpy's own scalar types explicitly at the point of computation.

**Q50. How would you approach adding types to an existing, large, wholly-untyped Jupyter-notebook-turned-production ETL script?**

Extract the notebook's logic into ordinary, importable, fully-typed functions and modules first (notebooks themselves are a poor target for static type checking); then apply the gradual-typing strategy from Part 10.7 to the new module.

### AI Engineer (Q51-Q60)

**Q51. How would you type a chat message list for an LLM conversation?**

As list[ChatMessage], where ChatMessage is a dataclass or TypedDict with a role: Literal["system", "user", "assistant"] field and a content: str field, exactly as shown in Part 11.5 and Project 8.

**Q52. How do you model an LLM's decision to either reply directly or call a tool?**

As a discriminated union of TypedDicts (or dataclasses) sharing a common tag field with distinct Literal values (e.g. kind: Literal["direct_reply"] vs kind: Literal["tool_call"]), letting the checker narrow correctly once the tag is checked - the approach used in Project 8.

**Q53. Why is Protocol especially common in AI-engineering pipelines (loaders, chunkers, retrievers)?**

AI pipelines are frequently composed of swappable stages sourced from different libraries or custom code (a local file loader vs. an S3 loader, for instance); Protocol lets every stage plug into the pipeline structurally, without forcing every implementation to inherit from a shared base class.

**Q54. What is the correct way to type a streaming LLM response function?**

As an async generator returning AsyncIterator[StreamChunk] (or AsyncGenerator[StreamChunk, None]), where StreamChunk is a small typed object carrying at least the text delta and a completion flag, as shown in Part 13.5.

**Q55. Why should a raw LLM JSON response never be trusted purely because it matches your TypedDict annotation?**

TypedDict provides zero runtime enforcement - annotations are a static-only promise (Part 10). An LLM's actual output is untrusted external data and should be validated at runtime, typically with Pydantic, before your typed application code relies on its shape.

**Q56. How would you type an embedding vector, and what are the tradeoffs of your choice?**

list[float] is simple and portable but offers no performance benefit; NDArray[np.float32] (or float64) is faster and more memory-efficient for large-scale vector operations but couples the type to numpy - the choice depends on whether the embeddings are consumed by numpy-based vector math or serialized as plain JSON.

**Q57. How do you type a tool/function-calling schema so it stays in sync with your actual Python tool implementations?**

Define a TypedDict (ToolDefinition, as in Part 11.5) describing the JSON schema your tool-calling API expects, and, where possible, generate it FROM your typed tool function's own signature (via introspection) rather than maintaining the schema and the implementation as two separately-drifting sources of truth.

**Q58. What typing challenge is unique to a ToolRegistry that dispatches calls to arbitrarily different tool functions?**

Each registered tool function may have a completely different parameter list and return type, which a single Callable[..., R] annotation cannot precisely capture per-tool - Project 8 handles this pragmatically with a scoped, justified type: ignore, while a fuller solution would use overload per known tool or a more advanced typed-registry pattern.

**Q59. How would you type a vector-search result ranking function?**

def top_k_results(results: list[SearchResult], k: int) -> list[SearchResult]:, where SearchResult is a small dataclass pairing a Document with a float score, as shown in Part 11.5 - keeping the function generic-free here is appropriate since the shape is fixed and well known.

**Q60. Why is NewType particularly useful for AI-engineering identifiers like document IDs and session IDs?**

AI applications typically juggle many string-based IDs (document IDs, session IDs, request IDs, user IDs) that are all technically str at runtime - NewType prevents accidentally passing one kind of ID where another was expected, exactly as demonstrated with AccountId in Project 3.

## Part 19 - 50 Practical Exercises

All 50 exercise prompts are listed first, grouped by difficulty. Complete solutions with explanations follow afterward in a dedicated Solutions Section, numbered to match.

### Beginner Exercises (1-15)

1. Annotate three variables: a person's name (str), age (int), and height in meters (float).

2. Write a function add(a, b) with parameter and return annotations, assuming both are int.

3. Write a function that takes a name (str) and returns a greeting (str).

4. Annotate a function log(message) that prints a message and returns nothing.

5. Annotate a list of five favorite movie titles.

6. Annotate a dictionary mapping product names (str) to prices (float).

7. Write a function that accepts a list[int] and returns their sum as an int.

8. Annotate a variable that may hold a username (str) or be unset (None).

9. Write a function get_discount(is_member: bool) -> float that returns 0.1 or 0.0.

10. Annotate a tuple representing an RGB color: three integers.

11. Annotate a set of unique tags (strings).

12. Write a function that takes an Optional[str] and returns "Unknown" if it is None, otherwise the value uppercased.

13. Annotate a class Book with title: str and pages: int as class-level annotations.

14. Write a function divide(a: float, b: float) -> float | None that returns None on division by zero instead of raising.

15. Run keyword.iskeyword() (from the earlier Keywords handbook) is not needed here - instead, run mypy on any file above and fix the first error it reports, if any.

### Intermediate Exercises (16-30)

1. Create a TypeAlias named Grade for Literal["A", "B", "C", "D", "F"], and use it in a function signature.

2. Write a generic function first_or_default(items: list[T], default: T) -> T using TypeVar.

3. Write a TypedDict UserProfile with required name and email fields, and an optional (NotRequired) bio field.

4. Write a NamedTuple Coordinates with x and y float fields, then create and unpack an instance.

5. Write a frozen dataclass Money with amount: float and currency: str fields.

6. Write a Protocol Readable with a read() -> str method, and a function that accepts any Readable.

7. Write a function that accepts a Callable[[int, int], int] and applies it to two numbers.

8. Write a function using Union (or |) that accepts either an int or a str ID and returns a normalized str.

9. Add a ClassVar counter to a class that increments every time a new instance is created.

10. Write a function using Final to declare a module-level constant MAX_USERS: Final[int] = 1000.

11. Write a function that narrows a str | None parameter using an if check, then safely calls .strip() on it.

12. Write a TypeGuard function is_list_of_int(values: list[object]) -> TypeGuard[list[int]].

13. Write a recursive TypedDict or type alias representing a simple nested category tree (name + list of subcategories).

14. Write a generic Stack[T] class with push and pop methods (see Part 5 for reference, but write it from scratch).

15. Write a function using @overload that returns int when given a str, and list[int] when given a list[str].

### Advanced Exercises (31-42)

1. Write a decorator using ParamSpec and TypeVar that measures and prints a function's execution time, preserving its original signature.

2. Write a decorator using Concatenate that injects a database connection object as the first argument to the wrapped function.

3. Write a NewType for OrderId and ProductId (both backed by int), and a function that would fail a type check if the two were swapped.

4. Model a discriminated union of two TypedDicts (Success and Failure) sharing a status: Literal field, and write a function that narrows correctly on each.

5. Write a generic Repository[T] class (as in Part 5.10) with add and get methods, then instantiate it for a custom Product class.

6. Write an Annotated type alias PositiveInt = Annotated[int, "must be > 0"], and use it in a function signature.

7. Write an async function using AsyncIterator[int] that yields numbers from 1 to n with a short asyncio.sleep between each.

8. Write a function annotated with NoReturn that always raises a custom exception, and use it inside another function to help narrow an Optional value.

9. Write a Protocol-based Strategy pattern implementation for two different sorting strategies (ascending/descending) applied to a list[int].

10. Write a function using cast() to satisfy a type checker after a runtime check that the checker cannot follow on its own, and explain in a comment why cast() was chosen over a real conversion.

11. Write a small typed ETL pipeline (extract/transform/load functions) using type aliases for each stage's data shape.

12. Write a Self-typed fluent QueryBuilder class with at least three chainable methods (where, order_by, limit).

### Expert / Challenge Exercises (43-50)

1. Design and implement a fully typed mini plugin system: a Protocol interface, a registry keyed by name, and at least two plugin implementations that satisfy the Protocol structurally (no inheritance).

2. Implement a typed Result[T, E] generic class (a lightweight Rust-style result type) with is_ok(), unwrap(), and a TypeGuard-friendly narrowing method, then use it in a function that can fail.

3. Build a typed LLM tool-calling schema generator: write a function that inspects a plain typed Python function's signature (using the inspect module) and produces a ToolDefinition TypedDict describing its parameters.

4. Implement a generic, typed in-memory Cache[K, V] with a maximum size and least-recently-used eviction, fully annotated including its internal data structures.

5. Write a strict-mode-compliant version of Project 4's ETL pipeline (Part 16) that adds proper error handling using a discriminated-union Result type at every stage instead of letting exceptions propagate.

6. Design a typed async pipeline that fetches data from three different Protocol-typed sources concurrently (using asyncio.gather) and merges the results into a single typed report object.

7. Take any function you previously typed with Any in an earlier exercise, and refactor it toward a fully precise type using TypeVar, Protocol, or a TypedDict/dataclass as appropriate - document what changed and why in a comment.

8. Configure a real pyproject.toml for a small multi-module project (at least 3 files) with mypy strict mode enabled, deliberately introduce two different kinds of type errors, run mypy, and paste (in a comment) the exact error messages produced.

## Solutions Section

Full solutions for all 50 exercises, in the same order and numbering as above, each with a short explanation of the key typing decision involved.

### Beginner Solutions (1-15)

In [ ]:
# 1
name: str = "Ada"
age: int = 34
height: float = 1.68

# 2
def add(a: int, b: int) -> int:
    return a + b

# 3
def greet(name: str) -> str:
    return f"Hello, {name}!"

# 4
def log(message: str) -> None:
    print(message)

# 5
favorite_movies: list[str] = ["Inception", "Dune", "Arrival"]

# 6
prices: dict[str, float] = {"widget": 4.99, "gadget": 12.5}

# 7
def total(values: list[int]) -> int:
    return sum(values)

# 8
username: str | None = None

# 9
def get_discount(is_member: bool) -> float:
    return 0.1 if is_member else 0.0

# 10
color: tuple[int, int, int] = (255, 0, 0)

# 11
tags: set[str] = {"python", "typing"}

# 12
def normalize(value: str | None) -> str:
    if value is None:
        return "Unknown"
    return value.upper()

# 13
class Book:
    title: str
    pages: int

# 14
def divide(a: float, b: float) -> float | None:
    if b == 0:
        return None
    return a / b

# 15 - no single code answer: run `mypy your_file.py` and fix whatever it reports.

Key idea across all fifteen: parameters and return types are annotated with the simplest, most honest type that matches actual behavior - notably #14, where the real possibility of returning None makes float | None the only accurate annotation, not float alone.

### Intermediate Solutions (16-30)

In [ ]:
# 16
from typing import Literal, TypeAlias
Grade: TypeAlias = Literal["A", "B", "C", "D", "F"]

def record_grade(grade: Grade) -> None:
    print(f"Recorded: {grade}")

# 17
from typing import TypeVar
T = TypeVar("T")

def first_or_default(items: list[T], default: T) -> T:
    return items[0] if items else default

# 18
from typing import TypedDict, NotRequired
class UserProfile(TypedDict):
    name: str
    email: str
    bio: NotRequired[str]

# 19
from typing import NamedTuple
class Coordinates(NamedTuple):
    x: float
    y: float

point = Coordinates(1.0, 2.0)
x, y = point

# 20
from dataclasses import dataclass
@dataclass(frozen=True)
class Money:
    amount: float
    currency: str

# 21
from typing import Protocol
class Readable(Protocol):
    def read(self) -> str: ...

def print_contents(source: Readable) -> None:
    print(source.read())

# 22
from typing import Callable
def apply_op(a: int, b: int, op: Callable[[int, int], int]) -> int:
    return op(a, b)

# 23
def normalize_id(raw: int | str) -> str:
    return str(raw)

# 24
from typing import ClassVar
class Widget:
    instance_count: ClassVar[int] = 0
    def __init__(self) -> None:
        Widget.instance_count += 1

# 25
from typing import Final
MAX_USERS: Final[int] = 1000

# 26
def shout(name: str | None) -> str:
    if name is not None:
        return name.strip().upper()
    return ""

# 27
from typing import TypeGuard
def is_list_of_int(values: list[object]) -> TypeGuard[list[int]]:
    return all(isinstance(v, int) for v in values)

# 28
from typing import TypedDict
class Category(TypedDict):
    name: str
    subcategories: list["Category"]

# 29
from typing import TypeVar, Generic
T = TypeVar("T")
class Stack(Generic[T]):
    def __init__(self) -> None:
        self._items: list[T] = []
    def push(self, item: T) -> None:
        self._items.append(item)
    def pop(self) -> T:
        return self._items.pop()

# 30
from typing import overload
@overload
def wrap(value: str) -> int: ...
@overload
def wrap(value: list[str]) -> list[int]: ...
def wrap(value: str | list[str]) -> int | list[int]:
    if isinstance(value, str):
        return len(value)
    return [len(v) for v in value]

Key idea across all fifteen: each exercise introduces exactly one new typing tool (TypeAlias, TypeVar, TypedDict, NamedTuple, frozen dataclass, Protocol, Callable, ClassVar, Final, narrowing, TypeGuard, recursive types, Generic, overload) in the smallest possible realistic shape, matching how each was first introduced earlier in this handbook.

### Advanced Solutions (31-42)

In [ ]:
# 31
from typing import Callable, ParamSpec, TypeVar
import time, functools
P = ParamSpec("P")
R = TypeVar("R")
def timed(func: Callable[P, R]) -> Callable[P, R]:
    @functools.wraps(func)
    def wrapper(*args: P.args, **kwargs: P.kwargs) -> R:
        start = time.perf_counter()
        result = func(*args, **kwargs)
        print(f"{func.__name__}: {time.perf_counter() - start:.4f}s")
        return result
    return wrapper

# 32
from typing import Callable, Concatenate, ParamSpec, TypeVar
P = ParamSpec("P")
R = TypeVar("R")
class DbConnection: ...
def with_connection(func: Callable[Concatenate[DbConnection, P], R]) -> Callable[P, R]:
    def wrapper(*args: P.args, **kwargs: P.kwargs) -> R:
        conn = DbConnection()
        return func(conn, *args, **kwargs)
    return wrapper

# 33
from typing import NewType
OrderId = NewType("OrderId", int)
ProductId = NewType("ProductId", int)
def ship_order(order_id: OrderId) -> None: ...
ship_order(ProductId(5))   # type checker error: ProductId is not OrderId

# 34
from typing import TypedDict, Literal, Union
class Success(TypedDict):
    status: Literal["success"]
    data: str
class Failure(TypedDict):
    status: Literal["failure"]
    error: str
Result = Union[Success, Failure]
def handle(result: Result) -> str:
    if result["status"] == "success":
        return result["data"]
    return f"Error: {result['error']}"

# 35
from typing import TypeVar, Generic
T = TypeVar("T")
class Repository(Generic[T]):
    def __init__(self) -> None:
        self._items: dict[int, T] = {}
        self._next_id = 1
    def add(self, item: T) -> int:
        self._items[self._next_id] = item
        self._next_id += 1
        return self._next_id - 1
    def get(self, item_id: int) -> T | None:
        return self._items.get(item_id)

# 36
from typing import Annotated
PositiveInt = Annotated[int, "must be > 0"]
def set_quantity(quantity: PositiveInt) -> None: ...

# 37
from typing import AsyncIterator
import asyncio
async def count_up(n: int) -> AsyncIterator[int]:
    for i in range(1, n + 1):
        await asyncio.sleep(0.1)
        yield i

# 38
from typing import NoReturn
def fail(msg: str) -> NoReturn:
    raise ValueError(msg)
def get_length(value: str | None) -> int:
    if value is None:
        fail("value required")
    return len(value)

# 39
from typing import Protocol
class SortStrategy(Protocol):
    def sort(self, values: list[int]) -> list[int]: ...
class Ascending:
    def sort(self, values: list[int]) -> list[int]:
        return sorted(values)
class Descending:
    def sort(self, values: list[int]) -> list[int]:
        return sorted(values, reverse=True)
def apply_sort(values: list[int], strategy: SortStrategy) -> list[int]:
    return strategy.sort(values)

# 40
from typing import cast
def get_setting(key: str) -> object:
    return {"retries": 3}[key]
# We've verified 'retries' always stores an int via prior validation logic
# that the checker cannot see, so cast() communicates that verified trust:
retries = cast(int, get_setting("retries"))

# 41
type RawRow = dict[str, str]
type CleanRow = dict[str, float]
def extract() -> list[RawRow]:
    return [{"amount": "10.5"}]
def transform(rows: list[RawRow]) -> list[CleanRow]:
    return [{"amount": float(r["amount"])} for r in rows]
def load(rows: list[CleanRow]) -> None:
    print(rows)

# 42
from typing import Self
class QueryBuilder:
    def __init__(self) -> None:
        self.parts: list[str] = []
    def where(self, condition: str) -> Self:
        self.parts.append(f"WHERE {condition}")
        return self
    def order_by(self, column: str) -> Self:
        self.parts.append(f"ORDER BY {column}")
        return self
    def limit(self, n: int) -> Self:
        self.parts.append(f"LIMIT {n}")
        return self

Key idea across all twelve: each solution combines two or three typing tools together (ParamSpec with Concatenate, NewType with function boundaries, TypedDict unions with Literal tags), mirroring how professional code rarely uses one typing feature in isolation.

### Expert / Challenge Solutions (43-50)

In [ ]:
# 43 - Plugin system
from typing import Protocol

class Plugin(Protocol):
    def run(self, data: str) -> str: ...

class UppercasePlugin:
    def run(self, data: str) -> str:
        return data.upper()

class ReversePlugin:
    def run(self, data: str) -> str:
        return data[::-1]

class PluginRegistry:
    def __init__(self) -> None:
        self._plugins: dict[str, Plugin] = {}
    def register(self, name: str, plugin: Plugin) -> None:
        self._plugins[name] = plugin
    def run(self, name: str, data: str) -> str:
        return self._plugins[name].run(data)

In [ ]:
# 44 - Result[T, E]
from typing import TypeVar, Generic, TypeGuard

T = TypeVar("T")
E = TypeVar("E")

class Result(Generic[T, E]):
    def __init__(self, value: T | None = None, error: E | None = None) -> None:
        self._value = value
        self._error = error

    def is_ok(self) -> bool:
        return self._error is None

    def unwrap(self) -> T:
        if self._error is not None:
            raise ValueError(f"Called unwrap on an error result: {self._error}")
        assert self._value is not None
        return self._value

def safe_divide(a: float, b: float) -> Result[float, str]:
    if b == 0:
        return Result(error="division by zero")
    return Result(value=a / b)

In [ ]:
# 45 - Tool schema generator (simplified)
import inspect
from typing import TypedDict, Callable, Any

class ToolParam(TypedDict):
    name: str
    type: str

class ToolDefinition(TypedDict):
    name: str
    parameters: list[ToolParam]

def generate_tool_definition(func: Callable[..., Any]) -> ToolDefinition:
    signature = inspect.signature(func)
    params: list[ToolParam] = []
    for param_name, param in signature.parameters.items():
        annotation = param.annotation
        type_name = getattr(annotation, "__name__", str(annotation))
        params.append({"name": param_name, "type": type_name})
    return {"name": func.__name__, "parameters": params}

def get_weather(city: str, days: int) -> str:
    return f"Forecast for {city} over {days} days"

print(generate_tool_definition(get_weather))

In [ ]:
# 46 - Typed LRU cache
from typing import TypeVar, Generic
from collections import OrderedDict

K = TypeVar("K")
V = TypeVar("V")

class LruCache(Generic[K, V]):
    def __init__(self, max_size: int) -> None:
        self._max_size = max_size
        self._data: OrderedDict[K, V] = OrderedDict()

    def get(self, key: K) -> V | None:
        if key not in self._data:
            return None
        self._data.move_to_end(key)
        return self._data[key]

    def put(self, key: K, value: V) -> None:
        if key in self._data:
            self._data.move_to_end(key)
        self._data[key] = value
        if len(self._data) > self._max_size:
            self._data.popitem(last=False)

47-50 are open-ended, project-scale exercises without a single canonical solution - they extend Project 4 (ETL, Part 16), a three-source async merge, a prior exercise's refactor, and a real pyproject.toml/mypy run respectively. Verify each by actually running mypy --strict (or pyright) against your solution and confirming zero unexpected errors before considering it complete.

## Part 20 - Final Type-Hint Cheat Sheet

A single, comprehensive reference table covering every major feature in this handbook - syntax, meaning, and a minimal example - organized by category for quick lookup.

### Basic Annotations

| Feature | Syntax | Meaning / Example |
| --- | --- | --- |
| Variable annotation | name: str = "Ada" | Declares the intended type of a variable |
| Function parameters/return | def f(x: int) -> str: | Documents expected input and output types |
| None return | def log() -> None: | Function returns nothing meaningful |
| Class attribute | class C: x: int | Documents an attribute's type on the class |

### Generics (Built-in Collections)

| Feature | Modern Syntax (3.9+) | Legacy Syntax (pre-3.9) |
| --- | --- | --- |
| List | list[str] | List[str] (from typing import List) |
| Dict | dict[str, int] | Dict[str, int] |
| Fixed tuple | tuple[int, str] | Tuple[int, str] |
| Variable-length tuple | tuple[int, ...] | Tuple[int, ...] |
| Set | set[str] | Set[str] |

### Optional and Union

| Feature | Modern Syntax (3.10+) | Legacy Syntax |
| --- | --- | --- |
| Optional (value or None) | str \| None | Optional[str] (from typing import Optional) |
| Union of types | int \| float | Union[int, float] (from typing import Union) |

### Any, Literal, and Type Aliases

| Feature | Syntax | Meaning |
| --- | --- | --- |
| Any | from typing import Any | Disables static checking for that value entirely |
| Literal | Literal["GET", "POST"] | Restricts to specific exact values, not a whole type |
| Type alias (legacy) | UserId: TypeAlias = int | Reusable name for a type expression |
| Type alias (3.12+) | type UserId = int | Modern, preferred alias syntax; supports generics |

### Generics: TypeVar and Generic Classes

| Feature | Syntax | Meaning |
| --- | --- | --- |
| Basic TypeVar | T = TypeVar("T") | A reusable type placeholder |
| Bounded TypeVar | TypeVar("T", bound=Animal) | T must be Animal or a subclass |
| Constrained TypeVar | TypeVar("T", str, int) | T must be exactly one of the listed types |
| Generic function (legacy) | def f(x: list[T]) -> T: | Preserves input/output type relationship |
| Generic function (3.12+) | def f[T](x: list[T]) -> T: | Modern syntax, no separate TypeVar object |
| Generic class | class Box(Generic[T]): | A class parameterized over T |

### Callable and Protocol

| Feature | Syntax | Meaning |
| --- | --- | --- |
| Callable | Callable[[int, int], int] | A function taking two ints, returning an int |
| Callable, any params | Callable[..., int] | Any parameters, but returns int |
| Protocol | class P(Protocol): def m(self) -> str: ... | Structural interface - no inheritance required |

### TypedDict, NamedTuple, Dataclasses

| Feature | Syntax | Meaning |
| --- | --- | --- |
| TypedDict | class M(TypedDict): title: str | A dict with fixed, named, typed keys |
| Optional TypedDict field | field: NotRequired[float] | That key may be omitted (3.11+) |
| NamedTuple | class P(NamedTuple): x: float; y: float | Immutable, named-field tuple |
| Dataclass | @dataclass class C: x: int | Auto-generates __init__, __repr__, __eq__ |
| Mutable dataclass default | items: list[str] = field(default_factory=list) | A fresh mutable object per instance |
| Frozen dataclass | @dataclass(frozen=True) | Immutable after construction |

### ClassVar, Final, Self

| Feature | Syntax | Meaning |
| --- | --- | --- |
| ClassVar | count: ClassVar[int] = 0 | Belongs to the class, not each instance |
| Final variable | MAX: Final[int] = 100 | Static-checker error on reassignment |
| Final class/method | @final | Static-checker error on subclassing/overriding |
| Self | def add(self) -> Self: | Returns the actual calling class, for chaining |

### Narrowing, TypeGuard, Forward References

| Feature | Syntax | Meaning |
| --- | --- | --- |
| isinstance narrowing | if isinstance(x, int): | Narrows x to int within the branch |
| None narrowing | if x is not None: | Narrows away the None member of a union |
| TypeGuard | def f(x) -> TypeGuard[list[int]]: | Custom narrowing function (true branch only) |
| TypeIs (3.13+) | def f(x) -> TypeIs[str]: | Custom narrowing, both branches |
| Forward reference | next: "Node \| None" | References a not-yet-defined class by string |

### Static Type Checkers

| Tool | Install | Run |
| --- | --- | --- |
| Mypy | pip install mypy | mypy path/ |
| Pyright | pip install pyright | pyright path/ |
| Pylance | VS Code extension | Runs automatically in-editor (bundles Pyright) |
| Ruff (lint-level typing rules) | pip install ruff | ruff check path/ |

### Decorators: ParamSpec, Concatenate, Annotated

| Feature | Syntax | Meaning |
| --- | --- | --- |
| ParamSpec | P = ParamSpec("P") | Captures an entire parameter list as one unit |
| Using ParamSpec | def d(f: Callable[P, R]) -> Callable[P, R]: | Fully preserves the wrapped function's signature |
| Concatenate | Callable[Concatenate[str, P], R] | An extra leading parameter, plus a ParamSpec |
| Annotated | Annotated[int, "must be positive"] | Attaches inert metadata a library may read |

### Async Typing

| Feature | Syntax | Meaning |
| --- | --- | --- |
| Async function return | async def f() -> str: | Calling it produces an awaitable str |
| Awaitable | Awaitable[str] | Anything usable with await, yielding a str |
| Async iterator | AsyncIterator[int] | Usable in an async for loop, yielding int |
| Async generator | AsyncGenerator[int, None] | An async def function using yield |

### Exceptions, NewType, Overload, Cast

| Feature | Syntax | Meaning |
| --- | --- | --- |
| NoReturn | def fail() -> NoReturn: | Function never returns normally (always raises) |
| NewType | UserId = NewType("UserId", int) | A distinct, non-interchangeable flavor of int |
| overload | @overload def f(x: str) -> int: ... | Declares a precise per-input-type signature |
| cast | cast(int, value) | Static-only type promise - ZERO runtime effect |

### The One Rule That Governs Everything Above

Type hints are read by tools you choose to run - a static type checker, an IDE, or a runtime-validation library like Pydantic that deliberately re-implements enforcement on top of the same syntax. Python's interpreter itself never consults an annotation to accept or reject a value. Every feature in this cheat sheet exists to make that voluntary, tool-driven verification as precise, expressive, and maintainable as possible - not to change what the language does when your program actually runs.